# Labelled examples

This exploratory notebook serves for the creation of labelled examples

In [1]:
import pandas as pd
import json
from collections import Counter
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
import re
import pycountry
from src.text_processing_functions import *
from src.LLM_functions import *
import copy as cp

from src.data import *

In [2]:
#helper functions
def sort_appeal_type(x):
    pref_order = ["DREF Operation Final Report", "DREF Operation Update", "DREF Operation", "Operations Update"]
    for appealtype in pref_order:
        if appealtype in list(x['appealType']):
            return x[x['appealType'] == appealtype]

def select_report_by_appealCode(appeal_code, report_df):
    report = report_df.where(report_df.appealCode == appeal_code).dropna()
    return report

def print_report(appeal_code, report, text_field="nathaz_text"):
    print(f"{appeal_code}: {report.date}")
    text = report[text_field]
    print("\n".join(text))

def add_report_date(report, impact_dict_or_list):
    if isinstance(impact_dict_or_list, dict):
        impact_dict_or_list["reportDate"] = report.date
    elif isinstance(impact_dict_or_list, list):
        for i in range(len(impact_dict_or_list)):
            impact_dict_or_list[i]["reportDate"] = report.date
    else:
        raise TypeError("impact_dict_or_list must be a dictionary or a list of dictionaries")
    return impact_dict_or_list

def download_report(report, savelocation):
    link = report["reportLink"]
    savename = report["origType"]+".pdf"
    r = requests.get(link)
    with open(savelocation + savename, 'wb') as f:
        f.write(r.content)

def report_dict_to_df(labelled_reports_dict):
    df_list = []
    for k,v in labelled_reports_dict.items():
        df = pd.DataFrame(v)
        df['appealCode'] = k
        df_list.append(df)
    df_all = pd.concat(df_list)
    df_all.reset_index(inplace=True, drop=True)
    return df_all

In [11]:
#Load data
file_path = DATA_IN_JSONS + 'filtered_report_types_nat_hazards_nathaz_text.json'#'all_ifrc_reports_info_processed_extended.json' #'all_ifrc_reports_info_processed_extended_format_nb_std_units.json'

# Open and read the JSON file
with open(file_path, 'r') as json_file:
    filtered_reports = json.load(json_file)
filtered_reports = pd.DataFrame(filtered_reports)

In [18]:
filtered_reports.loc[filtered_reports.appealCode == "MDRBD022"]

,reportName,disasterType,dateTime,date,reportLink,location,appealCode,appealType,origType,pdfDownloaded,text,docName,disasterTypeReclassified,naturalHazard,text_processed,sentences,iso_code,hazards_found_kw,nathaz_text,secondaryDisasterType
996,Bangladesh - Monsoon Floods (MDRBD022),-,2020-12-05T08:00:00+01:00,05/12/2020,https://adore.ifrc.org/Download.aspx?FileId=36...,Bangladesh,MDRBD022,Operations Update,Operations Update 4,1,\nIFRC Internal \nEmergency Appeal n° MDRBD...,MDRBD022_Operations_Update_Bangladesh_Download...,Flood,1,IFRC Internal Emergency Appeal n MDRBD022 GLID...,[IFRC Internal Emergency Appeal n MDRBD022 GLI...,BGD,"[Mass movement, Flood, Extreme temperature, St...",[SITUATION ANALY SIS Description of the disast...,NaN
1105,Bangladesh - Monsoon Floods (MDRBD022),Flood,2020-06-02T12:00:00+02:00,02/06/2020,https://adore.ifrc.org/Download.aspx?FileId=30...,Bangladesh,MDRBD022,Operations Update,Operations Update 4,1,\nEmergency Appeal n° MDRBD022 GLIDE n° FL...,MDRBD022_Operations_Update_Bangladesh_Download...,Flood,1,Emergency Appeal n MDRBD022 GLIDE n FL2019 000...,[Emergency Appeal n MDRBD022 GLIDE n FL2019 00...,BGD,"[Mass movement, Flood, Extreme temperature]",[The Disaster Risk Reduction DRR activit ies h...,NaN
1219,Bangladesh - Monsoon Floods (MDRBD022),Flood,2019-11-26T10:00:00+01:00,26/11/2019,https://adore.ifrc.org/Download.aspx?FileId=27...,Bangladesh,MDRBD022,Operations Update,Operations Update 2,1,\nEmergency Appeal n° MDRBD022 GLIDE n° FL-...,MDRBD022_Operations_Update_Bangladesh_Download...,Flood,1,Emergency Appeal n MDRBD022 GLIDE n FL2019 000...,[Emergency Appeal n MDRBD022 GLIDE n FL2019 00...,BGD,"[Mass movement, Flood]",[SITUATION ANALYSIS Emergency Plan of Action O...,NaN
1251,Bangladesh - Monsoon Floods (MDRBD022),Flood,2019-09-16T10:00:00+02:00,16/09/2019,https://adore.ifrc.org/Download.aspx?FileId=25...,Bangladesh,MDRBD022,Operations Update,Operations Update 1,1,\nEmergency Appeal n° MDRBD022 Glide n° FL-2...,MDRBD022_Operations_Update_Bangladesh_Download...,Flood,1,Emergency Appeal n MDRBD022 Glide n FL2019 000...,[Emergency Appeal n MDRBD022 Glide n FL2019 00...,BGD,"[Wildfire, Mass movement, Flood]",[SITUATION ANALYSIS Description of the disa st...,NaN
1280,Bangladesh - Monsoon Floods (MDRBD022),Flood,2019-07-19T08:00:00+02:00,19/07/2019,https://adore.ifrc.org/Download.aspx?FileId=24...,Bangladesh,MDRBD022,DREF Operation,DREF Operation,1,\nDREF operation n° MDR BD022 Glide n° FL-2...,MDRBD022_DREF_Operation_Bangladesh_Download.aspx,Flood,1,DREF operation n MDR BD022 Glide n FL2019 0000...,[DREF operation n MDR BD022 Glide n FL2019 000...,BGD,"[Wildfire, Mass movement, Flood]",[DREF operation n MDR BD022 Glide n FL2019 000...,NaN


## Select reports to label

In [27]:
# select reports to be labelled
appealCode_list = ["MDRCN006", "MDRBD022", "MDRYE011", "MDRS2001", "MDRIQ014", "MDRGN015", "MDRSV012", "MDRMY003", "MDRBD015", "MDRKE058"]  #### UPDATE HERE WITH THE LIST OF APPEAL CODE THAT NEED TO BE LABELLED

reports_to_label_all = filtered_reports.loc[filtered_reports.appealCode.isin(appealCode_list)]
#filtered_reports.where(filtered_reports.appealCode.isin(appealCode_list)).dropna()

#filter by report type
reports_to_label_all = reports_to_label_all.groupby('appealCode', as_index=False).apply(lambda x: sort_appeal_type(x)).reset_index(drop=True)

#convert dates
reports_to_label_all.date = pd.to_datetime(reports_to_label_all.date, dayfirst=True)

#select most recent reports
reports_to_label = reports_to_label_all.groupby('appealCode', as_index=False).apply(lambda x: x.sort_values('date', ascending=False).head(1)).reset_index(drop=True)

#check that everything is there
print(f"Number appealCodes: {len(appealCode_list)}, number reports: {len(reports_to_label)}")
missing_appealCode = list(set(appealCode_list) - set(reports_to_label.appealCode))
print(f"Missing appealCodes: {missing_appealCode}")

Number appealCodes: 10, number reports: 9
Missing appealCodes: ['MDRBD015']


C:\Users\lhasbini\AppData\Local\Temp\ipykernel_8096\3942418395.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  reports_to_label_all = reports_to_label_all.groupby('appealCode', as_index=False).apply(lambda x: sort_appeal_type(x)).reset_index(drop=True)
C:\Users\lhasbini\AppData\Local\Temp\ipykernel_8096\3942418395.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  reports_to_label = reports_to_label_a

## Labelling

In [9]:
labelled_impact_reports_dict = {} # dict to store labelled reports
#empty dict structure to store results
# labelled_impact_reports_dict["appealCode"]=[
#     {"reportDate": None,
#      "impactSubtype" : None,
#      "impactValue" : None, 
#      "impactUnit" : None, 
#      "impactValuePrecision" : None, 
#      "impactValueMin" : None, 
#      "impactValueMax" : None, 
#      "annotation" : None,
#      "country" : None,
#      "location" : None,
#      "startYear" : None,
#      "startMonth" : None,
#      "startDay" : None,
#      "endYear" : None,
#      "endMonth" : None,
#      "endDay" : None,
#      "hazards" : None,
#     },
# ]

In [29]:
i=0
print(f"{reports_to_label.iloc[i].appealCode}: {reports_to_label.iloc[i].date}")
print(reports_to_label.iloc[i].reportLink)
reports_to_label.iloc[i].nathaz_text

MDRBD022: 2019-07-19 00:00:00
https://adore.ifrc.org/Download.aspx?FileId=247507


['DREF operation n MDR BD022 Glide n FL2019 000079 BGD Date of issue 18 July 2019 Expected timeframe 4 months Expected end date 18 November 2019 Category allocated to the of the disaster or crisis Orange DREF allocated CHF 452,439 Total n umber of people affected 2,176,519 Number of people to be assisted 50,000 Host National Society presence n of volunteers, staff, branches Bangladesh Red Crescent Society BDRCS over 575 Red Crescent volunteers and 100 staff mobili zed.',
 'Red Cross Red Crescent Movement part ners actively involved in the operation American Red Cross, British Red Cross, Danish R ed Cross, German Red Cross , Swedish Red Cross, Swiss Red Cross, Italian Red Cross, Turkish Red Crescent , Qatar Red Crescent and the International Committee of the Red C ross ICRC.',
 'Other partner organizations actively involved i n the operation Government of Bangladesh, UN RC, UNICEF, WFP, Terre das hommes TdH, Oxfam, START Network .',
 'A.',
 'Situation analysis Description of the disaste

In [33]:
labelled_impact_reports_dict["MDRBD022"]=[
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Affected People",
     "impactValue" : 2176519, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['DREF operation n MDRBD022 Glide n FL-2019-000079-BGD Date of issue 18 July 2019 Expected timeframe 4 months Expected end date 18 November 2019 Category allocated to the of the disaster or crisis Orange DREF allocated CHF 452,439 Total number of people affected 2,176,519 Number of people to be assisted 50,000 Host National Society presence (n of volunteers, staff, branches) Bangladesh Red Crescent Society (BDRCS) over 575 Red Crescent volunteers and 100 staff mobilized.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 18,
     "endYear" : 2019,
     "endMonth" : 11,
     "endDay" : 18,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 1000000, 
     "impactValueMax" : None, 
     "annotation" : ['While the monsoon season normally brings annual floods to the country and wider region, this year, widespread flooding in upstream countries, Nepal and India, where millions of people have been severely impacted, have meant that the scale of the flooding this year has been significantly exacerbated.'],
     "country" : ["Nepal", "India"],
     "location" : None,
     "startYear" : 2019,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 2100000, 
     "impactValueMax" : None, 
     "annotation" : ['According to National disaster response coordination centre (NDRCC) report dated 16 July more than 2.1 million people have been affected in 21 districts, around 100,000 houses destroyed and about 14,733 hectares of crop land damaged.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 100000, 
     "impactUnit" : "houses destroyed", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['According to National disaster response coordination centre (NDRCC) report dated 16 July more than 2.1 million people have been affected in 21 districts, around 100,000 houses destroyed and about 14,733 hectares of crop land damaged.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : 14733, 
     "impactUnit" : "hectares of crops", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['According to National disaster response coordination centre (NDRCC) report dated 16 July more than 2.1 million people have been affected in 21 districts, around 100,000 houses destroyed and about 14,733 hectares of crop land damaged.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"] 
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Other Infrastructural impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['It is also reported that embankments have been damaged and inundated in new areas.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Affected People",
     "impactValue" : 2176519, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of affected population 2,176,519 No.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 3988, 
     "impactUnit" : "houses fully damaged", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of fully damaged house 3,988 No.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 98571, 
     "impactUnit" : "houses partially damaged", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of partially damaged house 98,571 No.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Displaced People",
     "impactValue" : 31818, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of people who have moved to safe shelter 31,818 Amount of crop land damaged (Hectare) 14,733 No.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : 14733, 
     "impactUnit" : "hectares of crops", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of people who have moved to safe shelter 31,818 Amount of crop land damaged (Hectare) 14,733 No.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Other Infrastructural impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of union affected 776 Emergency Plan of Action (EPoA) Bangladesh Floods P a g e 2 People watch as water from the swollen Teesta river gush into their neighbourhood in Gaddimari area of Lalmonirhats Hatibandha after a part of the protection embankment collapsed on 13 July afternoon.'],
     "country" : ["Bangladesh"],
     "location" : ["Gaddimari area",  "Lalmonirhats Hatibandha"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 13,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }
]

In [37]:
i=1
print(f"{reports_to_label.iloc[i].appealCode}: {reports_to_label.iloc[i].date}")
print(reports_to_label.iloc[i].reportLink)
reports_to_label.iloc[i].nathaz_text

MDRCN006: 2019-03-14 00:00:00
https://adore.ifrc.org/Download.aspx?FileId=233003


['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.',
 'In some areas of North Central Sichuan, there were heavy rainstorms and torrential rains for four consecutive days.',
 'These were also compounded by the effects of two weather systems in the area Typhoon Prapiroon, and Typhoon Maria.',
 'Accordin g to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 need ed emergency relief in Sichuan prefectures of Deyang, Mianyang, G uangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.',
 'A total of 36,900 hectares of crops were also affected by the flood.',
 'The direct economic loss was estimated to be over 5.3 billion Yuan approxim ately CHF 792 million .',
 'Gansu province was hit e

In [40]:
labelled_impact_reports_dict["MDRCN006"]=[
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Affected People",
     "impactValue" : 1381000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Deyang", "Mianyang", "Guangyuan"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood", "Tropical storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 3, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Deyang", "Mianyang", "Guangyuan"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood", "Tropical storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Displaced People",
     "impactValue" : 222000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Deyang", "Mianyang", "Guangyuan"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood", "Tropical storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Injured People",
     "impactValue" : 22000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Deyang", "Mianyang", "Guangyuan"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood", "Tropical storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : "houses collapsed", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 900, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Deyang", "Mianyang", "Guangyuan"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood", "Tropical storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : "houses damaged", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 29000, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Deyang", "Mianyang", "Guangyuan"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood", "Tropical storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : 36900, 
     "impactUnit" : "hectares of crops", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['A total of 36,900 hectares of crops were also affected by the flood.'],
     "country" : ["China"],
     "location" : ["Sichuan", "southeast region of Gansu"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Other Economic and Livelihood Impacts",
     "impactValue" : None, 
     "impactUnit" : "CHF", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 792000000, 
     "impactValueMax" : None, 
     "annotation" : ['The direct economic loss was estimated to be over 5.3 billion Yuan approximately CHF 792 million.'],
     "country" : ["China"],
     "location" : ["Sichuan", "southeast region of Gansu"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Affected People",
     "impactValue" : 1519000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['A heavy rainstorm occurred in Southeast Gansu from 10 to 11 July 2018.', 
                     'The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 10,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 11,
     "hazards" : ["Flood", "Convective storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 12, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['A heavy rainstorm occurred in Southeast Gansu from 10 to 11 July 2018.', 
                     'The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 10,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 11,
     "hazards" : ["Flood", "Convective storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Missing People",
     "impactValue" : 4, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['A heavy rainstorm occurred in Southeast Gansu from 10 to 11 July 2018.', 
                     'The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 10,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 11,
     "hazards" : ["Flood", "Convective storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Displaced People",
     "impactValue" : 30000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['A heavy rainstorm occurred in Southeast Gansu from 10 to 11 July 2018.', 
                     'The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 10,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 11,
     "hazards" : ["Flood", "Convective storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : "houses collapsed", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 2300, 
     "impactValueMax" : None, 
     "annotation" : ['A heavy rainstorm occurred in Southeast Gansu from 10 to 11 July 2018.', 
                     'More than 2,300 houses collapsed, and 19,000 were damaged to varying degrees.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 10,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 11,
     "hazards" : ["Flood", "Convective storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : "houses damaged", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 19000, 
     "impactValueMax" : None, 
     "annotation" : ['A heavy rainstorm occurred in Southeast Gansu from 10 to 11 July 2018.', 
                     'More than 2,300 houses collapsed, and 19,000 were damaged to varying degrees.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 10,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 11,
     "hazards" : ["Flood", "Convective storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Other Economic and Livelihood Impacts",
     "impactValue" : 538000000, 
     "impactUnit" : "CHF", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['A heavy rainstorm occurred in Southeast Gansu from 10 to 11 July 2018.', 
                     'The direct economic loss was estimated 3.6 billion Yuan approximately CHF 538 million.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 10,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 11,
     "hazards" : ["Flood", "Convective storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : "houses", 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['According to this rapid assessment, heavy rainfall had resulted in a large number of seriously damaged houses that have continued to collapse in these two provinces.'],
     "country" : ["China"],
     "location" : ["Sichuan", "southeast region of Gansu"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }
]

In [52]:
i=2
print(f"{reports_to_label.iloc[i].appealCode}: {reports_to_label.iloc[i].date}")
print(reports_to_label.iloc[i].reportLink)
print(f"Number of sentences : {len(reports_to_label.iloc[i].nathaz_text)}")
reports_to_label.iloc[i].nathaz_text[:30]

MDRGN015: 2025-02-13 00:00:00
https://go-api.ifrc.org/api/downloadfile/85154/MDRGN015dfr.
Number of sentences : 37


['Appeal MDRGN015Total DREF Allocation Crisis Category YellowHazard Flood Glide Number FL2023000158GINPeople Affected 24,135 peoplePeople Targeted 14,350 peoplePeople Assisted Event Onset SuddenOperation Start Date 23082023Operational End Date Total Operating Timeframe 5 months Targeted AreasKindia Page 1 19Description of the Event Date of event 06082023 What happened, where and when?',
 'Guinea experienced persistent torrential rains from the beginning of August 2023.',
 'The highest recorded incidents occurred on Sunday, 6 August 2023 in Coyah, and on Friday, 11 August 2023 in Conakry and Siguiri.',
 'These rains caused associated impact, including flooding in lowlying areas and the overflow of rivers.',
 'Major roads in Conakry and Siguiri were rendered impassable due to the floodwaters, heavily constraining vehicles, and forcing pedestrians to find alternative routes.',
 'Different prefectures across the country continued to experience heavy rains, which led to flooding in addition

In [53]:
reports_to_label.iloc[i].nathaz_text[30:]

['Page 3 19The coping capacities of affected households had already been stretched, considering the floods had washed away all household items, including reserved food, compounded by the ongoing economic crisis in the country.',
 'As the situation remained fluid, the additional impact of flooding increased vulnerabilities and reduced the capacity to cope.',
 'The Guinea Meteorological Department had forecasted more heavy rain for several prefectures, which could result in flooding, especially in floodprone communities and areas near rivers where overflow was likely.',
 'The continuous flooding in these locations, coupled with the absence of support for the affected people, called for urgent efforts to address immediate humanitarian needs.',
 'The mayor of Coyah prefecture called for immediate support from humanitarian organizations and wellwishers.',
 'National Society Actions Have the National Society conducted any intervention additionally to those part of this DREF Operation?No Plea

In [ ]:
### NEED TO REDO LABELLING FOR MDRGN015

In [42]:
i=3
print(f"{reports_to_label.iloc[i].appealCode}: {reports_to_label.iloc[i].date}")
print(reports_to_label.iloc[i].reportLink)
reports_to_label.iloc[i].nathaz_text

MDRIQ014: 2023-05-12 00:00:00
https://go-api.ifrc.org/api/DownloadFile/65831/MDRIQ014fr


['The major donors and partners of the Disaster Relief Emergency Fund DREF include the Red Cross Societies and governments of Belgium, Britain, Canada, Denmark, German, Ireland, Italy, Japan, Luxembourg, New Zealand, Norway, the Republic of Korea, Spain, Sweden, and Switzerland, as well as DG ECHO and Blizzard Entertainment, Mondelez International Foundation, and Fortive Corporation and o ther corporate and private donors.',
 'The IFRC, on behalf of the Iraqi Red Crescent Society , would like to extend thanks to all for their generous contributions .',
 'A.',
 'SITUATION ANALYSIS Description of the disaster Iraq is at risk of multiple disasters ranging from natural phenomena such as drought, sandstorm s, heatwaves, and floods, to man made ones.',
 'After one of the driest years in decades, h eavy rains slammed Iraqs northern Kurdish region on 17 December 2021.',
 'The overnight rainfall caused a flash flood in Erbil, the regions capital, and the Kirkuk governorate in northern Iraq.',
 

In [55]:
labelled_impact_reports_dict["MDRIQ014"]=[
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 7500, 
     "impactValueMax" : None, 
     "annotation" : ['P a g e 1 Internal DREF Operation n MDRIQ014 Glide n FL-2021-000208-IRQ Date of issue 27/12/2021 Expected timeframe 4 months Expected end date 30/04/2022 Category allocated to the of the disaster or crisis Yellow DREF allocated CHF 225,874 Total number of people affected 7,500+ Number of people to be assisted 7,500 (1,250 families) Provinces affected Erbil & Kirkuk Provinces/Regions targeted Erbil & Kirkuk Operating National Society presence (n of volunteers, staff, branches) The Iraqi Red Crescent Society (IRCS) is a voluntary humanitarian organization IRCS has a strong branch network in the country, which is capable of providing relief in times of disasters/emergencies.'],
     "country" : ["Iraq"],
     "location" : ["Erbil", "Kirkuk"],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 27,
     "endYear" : 2022,
     "endMonth" : 4,
     "endDay" : 30,
     "hazards" : ["Flood", "Drought"]
    }, 
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The incident caused widespread damage to houses, infrastructure, and vehicles.', 
                     'The heavy rains overnight caused a flash flood in Erbil, the regions capital, and Kirkuk Governorate, located in Northern Iraq.', 
                     "On 17 December 2021, heavy rainfalls hit the country's northern Kurdish region.",],
     "country" : ["Iraq"],
     "location" : ["Erbil", "Kirkuk"],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 17,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Drought"]
    }, 
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Other Infrastructure impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The incident caused widespread damage to houses, infrastructure, and vehicles.', 
                     'The heavy rains overnight caused a flash flood in Erbil, the regions capital, and Kirkuk Governorate, located in Northern Iraq.', 
                     "On 17 December 2021, heavy rainfalls hit the country's northern Kurdish region.",],
     "country" : ["Iraq"],
     "location" : ["Erbil", "Kirkuk"],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 17,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Drought"]
    }, 
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Road Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The incident caused widespread damage to houses, infrastructure, and vehicles.', 
                     'The heavy rains overnight caused a flash flood in Erbil, the regions capital, and Kirkuk Governorate, located in Northern Iraq.', 
                     "On 17 December 2021, heavy rainfalls hit the country's northern Kurdish region.",],
     "country" : ["Iraq"],
     "location" : ["Erbil", "Kirkuk"],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 17,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood","Drought"]
    }, 
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ["Muddy water swept into peoples homes in Erbil's Daratu, Qushtapa, Shamamk, Zhyan, Roshinbiri, and Bahrka neighbourhoods in the early hours of the morning, forcing individuals out of their houses.", 
                     "On 17 December 2021, heavy rainfalls hit the country's northern Kurdish region."],
     "country" : ["Iraq"],
     "location" : ["Erbil's Daratu neighbourhood", "Qushtapa neighbourhood", "Shamamk neighbourhood", "Zhyan neighbourhood", "Roshinbiri neighbourhood", "Bahrka neighbourhood"],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 17,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Drought"]
    }, 
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 14, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['According to the Kurdish region government on 19 December, 14 people were reported dead by the floods and Emergency Plan of Action (EPoA) Iraq Flash Floods Figure 1 Flooding in Erbil governorate, Iraq (Photo IRCS) P a g e 2 Internal more than 7,000 people are affected by these floods, while IRCS carried further rapid assessments to confirm with the affected families.', 'IRCS carried out further rapid assessments to confirm the number of casualties and affected families , reaching a total of 14 casualties and 7,500 people affected 1,250 families .'],
     "country" : ["Iraq"],
     "location" : ["Kurdish region"],
     "startYear" : 2021,
     "startMonth" : 12, 
     "startDay" : 19,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Drought"]
    },  
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 7000, 
     "impactValueMax" : 7500, 
     "annotation" : ['According to the Kurdish region government on 19 December, 14 people were reported dead by the floods and Emergency Plan of Action (EPoA) Iraq Flash Floods Figure 1 Flooding in Erbil governorate, Iraq (Photo IRCS) P a g e 2 Internal more than 7,000 people are affected by these floods, while IRCS carried further rapid assessments to confirm with the affected families.', 
                     'IRCS carried out further rapid assessments to confirm the number of casualties and affected families , reaching a total of 14 casualties and 7,500 people affected 1,250 families .'],
     "country" : ["Iraq"],
     "location" : ["Kurdish region"],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 19,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Drought"]
    },  
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Missing People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['More are feared missing while search and rescue operations are ongoing.'],
     "country" : ["Iraq"],
     "location" : ["Kurdish region"],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 19,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Drought"]
    }, 
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Affected People",
     "impactValue" : 7000000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The dis astrous heavy rains came at a time when Iraq was already suffering from severe droughts, w ith seven million Iraqis already affected along with the majority of agricultural lands .'],
     "country" : ["Iraq"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Drought"]
    }, 
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The dis astrous heavy rains came at a time when Iraq was already suffering from severe droughts, w ith seven million Iraqis already affected along with the majority of agricultural lands .'],
     "country" : ["Iraq"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Drought"]
    }
]

In [57]:
i=4
print(f"{reports_to_label.iloc[i].appealCode}: {reports_to_label.iloc[i].date}")
print(reports_to_label.iloc[i].reportLink)
reports_to_label.iloc[i].nathaz_text[:30]

MDRKE058: 2023-11-15 00:00:00
https://adore.ifrc.org/Download.aspx?FileId=760340


['Page 1 21 DREF Operation Kenya Floods Evacuation of the a ected families in Garissa County Appeal MDRKE058Country KenyaHazard FloodType of DREF Response Crisis Category OrangeEvent Onset SlowDREF Allocation CHF 749,939 Glide Number People A ected 281,880 peoplePeople Targeted 150,000 people Operation Start Date 20231111Operation Timeframe 4 monthsOperation End Date 20240331DREF Published 20231115 Targeted Areas Tana River, Garissa, Wajir, Mandera, Marsabit, Isiolo, MeruPage 2 21 Description of the Event KRCS oods damage What happened, where and when?',
 'Kenya has been experiencing enhanced rains since September, with alarming level rainfall as a result of El Nino conditions and a positive Indian Ocean Dipole IOD which are currently present in Pacic Ocean and Indian Ocean respectively.',
 'The Kenya Meteorological Department KMD indicate the rains are expected to peak in November but may continue into January 2024.',
 'From beginning of November, the rst regions to experience signica

In [ ]:
reports_to_label.iloc[i].nathaz_text[30:]

['South Rift number of households aected 7 35 of which none displaced.',
 'Aerial view during the rapid evaluation have also identied that oods aected also El Wak of Mandira country with the population of 7,500 households was totally submerged.',
 '2 Vulnerabilities and other damages Livestock, agriculture, infrastructure 7,806 livestock deaths, 566 acreages destroyed, 26 boreholes ooded, 149 latrines destroyed, 15 items of infrastructure damaged or destroyed.',
 'Local health facilities, schools and other public oces have been submerged in water in most of the aected counties.',
 'In Northern Kenya, where communities have long displayed resilience in the face of water scarcity and hunger during droughts, residents now grapple with the challenges posed by rising oodwaters.',
 'Crops in large tracts of land have been submerged by the oods posing serious food shortages in the future.',
 'There is also seeing ooding in urban areas particularly informal settlements where there is uncontrol

In [ ]:
### NEED TO REDO LABELLING

In [63]:
i=5
print(f"{reports_to_label.iloc[i].appealCode}: {reports_to_label.iloc[i].date}")
print(reports_to_label.iloc[i].reportLink)
reports_to_label.iloc[i].nathaz_text[:30]

MDRMY003: 2017-11-21 00:00:00
https://adore.ifrc.org/Download.aspx?FileId=176204


['Situation analysis Description of the disaster Heavy rains that started in December 2016 continued until late January 2017 in parts of Malaysia, causing flooding in seven states of Peninsular Malaysia Johor, Kelantan, Pahang, Perak, Terengganu, Malacca and Selangor and Sabah in East Malaysia.',
 'More than 23,000 people, mainly from smaller towns and villages in rural areas, had to leave their homes to established relief centres .',
 'The situation improved significantly after the weekend of Lunar New Year 28 29 January, with floodwater receding in severa l affected districts, allowing families that were in relief centres to return home.',
 'National Agency for D isaster Administration NADMA reported that the state of Johor suffered the brunt of rising waters, with more than 8,000 evacuees and a fatality.',
 'More information on the floods can be obtained from Information Bulletin n1 issued on 5 January, Information Bulletin n2 issued on 27 January and Information Bulletin n3 issued 

In [64]:
reports_to_label.iloc[i].nathaz_text[30:]

['The strategy adopted To support MRCS in meeting the immediate needs of affected families, IFRC allocated CHF 73,239 from DREF on 10 February 2017 , with focus on provision of h ygiene kits and hygiene promotion.']

In [65]:
labelled_impact_reports_dict["MDRMY003"]=[
    {"reportDate": "2017-11-21",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 23000, 
     "impactValueMax" : None, 
     "annotation" : ['More than 23,000 people, mainly from smaller towns and villages in rural areas, had to leave their homes to established relief centres.', 
                     'Situation analysis Description of the disaster Heavy rains that started in December 2016 continued until late January 2017 in parts of Malaysia, causing flooding in seven states of Peninsular Malaysia Johor, Kelantan, Pahang, Perak, Terengganu, Malacca and Selangor and Sabah in East Malaysia.'],
     "country" : ["Malaysia"],
     "location" : ["Johor", "Kelantan", "Pahang", "Perak", "Terengganu", "Malacca", "Selangor", "Sabah"],
     "startYear" : 2016,
     "startMonth" : 12,
     "startDay" : None,
     "endYear" : 2017,
     "endMonth" : 1,
     "endDay" : 31,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2017-11-21",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 8000, 
     "impactValueMax" : None, 
     "annotation" : ['National Agency for Disaster Administration (NADMA) reported that the state of Johor suffered the brunt of rising waters, with more than 8,000 evacuees and a fatality.'],
     "country" : ["Malaysia"],
     "location" : ["Johor"],
     "startYear" : 2016,
     "startMonth" : 12,
     "startDay" : None,
     "endYear" : 2017,
     "endMonth" : 1,
     "endDay" : 31,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2017-11-21",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 1, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['National Agency for Disaster Administration (NADMA) reported that the state of Johor suffered the brunt of rising waters, with more than 8,000 evacuees and a fatality.'],
     "country" : ["Malaysia"],
     "location" : ["Johor"],
     "startYear" : 2016,
     "startMonth" : 12,
     "startDay" : None,
     "endYear" : 2017,
     "endMonth" : 1,
     "endDay" : 31,
     "hazards" : ["Flood"]
    }
]

In [68]:
i=6
print(f"{reports_to_label.iloc[i].appealCode}: {reports_to_label.iloc[i].date}")
print(reports_to_label.iloc[i].reportLink)
reports_to_label.iloc[i].nathaz_text[:30]

MDRS2001: 2024-11-26 00:00:00
https://adore.ifrc.org/Download.aspx?FileId=840377


['SITUATION ANALYSIS Description of the crisis Prior to Hurricane Beryl hitting Grenada on July 1, 2024, the Government of Grenada had declared a water crisis effective May 12, 2024 due to an acute shortage of water resources.',
 'Although the water use restrictions were lifted on June 18, 2024, people were still being affected by water regulation schedules as the water authority was not yet back to full capacity and normal operating conditions .',
 'Therefore, portions of the population, primarily in the south of Grenada, were already facing vulnerabilities related to water shortages.',
 'Grenada is exposed to several natural hazards and has historical experience being impacted by cyclones, floods, droughts, landslides, rock falls, earthquakes, forest fires, road accidents, and epidemics.',
 'Generally, natural disasters and climate change are existential threats to Grenad a, with annual losses from these events estimated at 1.7 percent of its GDP.',
 'Grenada is highly vulnerable acr

In [67]:
reports_to_label.iloc[i].nathaz_text[30:]

['In addition to the desalination plants, household water storage tanks and cisterns have either been destroyed or compromised, requiring replacement or water treatment.',
 'Impact on physical and mental wellbeing As over 95 of housing, livelihoods and assets have been affected, including clinics, daycares, hospitals, aged care homes, and social services, there is a need for psychosocial support, particularly for the elderly, children, people with disabilities, and other vulnerable groups.',
 'The government of Grenada deployed a psychosocial support team to Carriacou and Martinique, as well as northern Grenada, and assessments are currently being conducted together with the Ministry of Education, UNICEF, and UN Women.',
 'Due to the near total devastation in Carriacou and Petite Martinique, the elderly are being relocated to Grenada for basic needs and geriatric care.',
 'The World Food Program is reportedly implementing immediate cash transfers to vulnerable groups.',
 'The governmen

In [ ]:
#### REDO LABELLING

In [18]:
labelled_impact_reports_dict["MDRS2001"]=[
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 55, 
     "impactUnit" : "houses damaged", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Barbados was spared a direct hit from the hurricane, as it shifted direction less than 72 hours before landfall.', 
                     'Although 55 homes suffered minor damages, the fishing industry was particularly hard hit, with over 200 vessels damaged or destroyed, together with fishing industry infrastructure, disrupting the livelihoods of the coastal communities.'],
     "country" : ["Barbados"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 6,
     "startDay" : 25,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Agricultural Infrastructure",
     "impactValue" : 200, 
     "impactUnit" : "vessels", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Barbados was spared a direct hit from the hurricane, as it shifted direction less than 72 hours before landfall.', 
                     'Although 55 homes suffered minor damages, the fishing industry was particularly hard hit, with over 200 vessels damaged or destroyed, together with fishing industry infrastructure, disrupting the livelihoods of the coastal communities.'],
     "country" : ["Barbados"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 6,
     "startDay" : 25,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Displaced People",
     "impactValue" : 1600, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['On July 1, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.', 
                     'In Grenada, more than 1,600 people were forced into shelters, with 98% of buildings on Carriacou and Petit Martinique islands suffering severe damage.'],
     "country" : ["Grenada"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    },  
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Other Infrastructure impact",
     "impactValue" : 98, 
     "impactUnit" : "% buildings damaged", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['On July 1, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.', 
                     'In Grenada, more than 1,600 people were forced into shelters, with 98% of buildings on Carriacou and Petit Martinique islands suffering severe damage.'],
     "country" : ["Grenada"],
     "location" : ["Carriacou islands", "Petit Martinique island"],
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Healthcare Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['On July 1, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.', 
                     'The destruction extended to critical infrastructure, including healthcare facilities and the airport terminal, as well as electrical and water utilities.'],
     "country" : ["Grenada"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    },  
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Other Transportation Infrastructure",
     "impactValue" : None, 
     "impactUnit" : "airport", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 1, 
     "impactValueMax" : None, 
     "annotation" : ['On July 1, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.', 
                     'The destruction extended to critical infrastructure, including healthcare facilities and the airport terminal, as well as electrical and water utilities.'],
     "country" : ["Grenada"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Power and Energy Production Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['On July 1, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.', 
                     'The destruction extended to critical infrastructure, including healthcare facilities and the airport terminal, as well as electrical and water utilities.'],
     "country" : ["Grenada"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Water, Sanitation, and Hygiene Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['On July 1, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.', 
                     'The destruction extended to critical infrastructure, including healthcare facilities and the airport terminal, as well as electrical and water utilities.'],
     "country" : ["Grenada"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 90, 
     "impactUnit" : "% of homes damaged", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['On July 1, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.', 
                     'Saint Vincent and the Grenadines experienced similar devastation, with 90% of homes on Union Island damaged or destroyed, impacting essential services and leaving many residents without access to healthcare and other services.'],
     "country" : ["Saint Vincent and the Grenadines"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Access to Healthcare",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['On July 1, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.', 
                     'Saint Vincent and the Grenadines experienced similar devastation, with 90% of homes on Union Island damaged or destroyed, impacting essential services and leaving many residents without access to healthcare and other services.'],
     "country" : ["Saint Vincent and the Grenadines"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Other Infrastructure impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Jamaica experienced widespread damage as Beryl once again intensified to Category 5, severely impacting infrastructure and agriculture.'],
     "country" : ["Jamaica"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Other Agricultural impact",
     "impactValue" : 1000000000, 
     "impactUnit" : 'USD', 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Jamaica experienced widespread damage as Beryl once again intensified to Category 5, severely impacting infrastructure and agriculture.', 
                     'The agricultural sector alone suffered losses estimated at USD 1 billion, severely affecting food security and local economies.'],
     "country" : ["Jamaica"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Other Agricultural impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Jamaica experienced widespread damage as Beryl once again intensified to Category 5, severely impacting infrastructure and agriculture.', 
                     'The hardest-hit areas included Clarendon and St. Elizabeth, with extensive damage reported in St. Thomas, Manchester, Westmoreland, and Hanover.'],
     "country" : ["Jamaica"],
     "location" : ["Clarendon", "St. Elizabeth", "St. Thomas", "Manchester", "Westmoreland", "Hanover"],
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Other Infrastructure impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Jamaica experienced widespread damage as Beryl once again intensified to Category 5, severely impacting infrastructure and agriculture.', 
                     'The hardest-hit areas included Clarendon and St. Elizabeth, with extensive damage reported in St. Thomas, Manchester, Westmoreland, and Hanover.'],
     "country" : ["Jamaica"],
     "location" : ["Clarendon", "St. Elizabeth", "St. Thomas", "Manchester", "Westmoreland", "Hanover"],
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Other Economic & Livelihood impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The agricultural sector alone suffered losses estimated at USD 1 billion, severely affecting food security and local economies.'],
     "country" : ["Jamaica"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The destruction of homes and critical infrastructure has led to a significant displacement crisis, with many seeking refuge in temporary shelters.'],
     "country" : ["Jamaica"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The destruction of homes and critical infrastructure has led to a significant displacement crisis, with many seeking refuge in temporary shelters.'],
     "country" : ["Jamaica"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Water, Sanitation, and Hygiene Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['This displacement, combined with damaged water and sanitation facilities, poses a heightened risk of waterborne diseases, including leptospirosis and cholera.'],
     "country" : ["Jamaica"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['This displacement, combined with damaged water and sanitation facilities, poses a heightened risk of waterborne diseases, including leptospirosis and cholera.', 
                     'The psychological impact on survivors is also profound, with many experiencing trauma and stress, necessitating comprehensive mental health and psychosocial support services.'],
     "country" : ["Jamaica"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm", "Epidemic"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Recreation, Tourism, and Culture",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The economic toll of Hurricane Beryl is substantial, with extensive damage to key sectors, such as agriculture, fishing, and tourism.'],
     "country" : ["Jamaica", "Barbados", "Saint Vincent and the Grenadines"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }   
]

In [69]:
i=7
print(f"{reports_to_label.iloc[i].appealCode}: {reports_to_label.iloc[i].date}")
print(reports_to_label.iloc[i].reportLink)
reports_to_label.iloc[i].nathaz_text

MDRSV012: 2019-06-26 00:00:00
https://adore.ifrc.org/Download.aspx?FileId=243784


['The major donors and partners of the Disaster Relief Emergency Fund DREF include the Red Cross Societies and governments of Belgium, Britain, Canada, Denmark, German, Ireland, Italy, Japan, Luxembourg, New Zealand, Norway, Republic of Korea, Spain, Sweden and Switzerland, as well as DG ECHO and Blizzard Entertainment, Mondelez International Foundation, and Fortive Corporation and other corporate and private donors.',
 'The IFRC, on behalf of the national society, would like to extend thanks to all for their generous contributions.',
 'ECHO and the government of Canada have replenished the DREF in the occasion of this operation.',
 'The total amount spent under this DREF operation was 118,630 CHF.',
 'The remaining balance of 32,041 CHF will be reimbursed to the Disaster Relief Emergency Fund .',
 'For the Final Financial Report, click here .',
 'For contact information, click here .',
 'A.',
 'Situation analysis Description of the disaster On 6 October, rains began falling over easte

In [70]:
labelled_impact_reports_dict["MDRSV012"]=[
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.', 
                     'Affected transit routes TOTAL Flooded highways 1 Flooded roads 3 Affected highways 24 Affected rods 31 Isolated communities 1 Affected people TOTAL Injured 14 Dead 4 Sheltered 1,090 Active shelters 13 Other TOTAL Fallen trees 42 Branches of fallen trees 5 Landslides 74 Floods 1 Overflowed rivers 7 Subsidences 1 Vehicles directly affected by the event 5 Affected homes and buildings TOTAL Affected homes 6 Flooded homes 1,409 Destroyed homes 2 Other buildings affected 1 Other buildings destroyed collapsed walls 6 Summary of response Overview of Host National Society The Salvador ean Red Cross Society SRCS constantly monitored the situation through its branches across the country since the onset of lowpressure system No .'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Road Infrastructure",
     "impactValue" : 24, 
     "impactUnit" : 'highways', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Road Infrastructure",
     "impactValue" : 31, 
     "impactUnit" : 'roads', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Injured People",
     "impactValue" : 14, 
     "impactUnit" : 'people', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 1, 
     "impactUnit" : 'people', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : 42, 
     "impactUnit" : 'trees', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 4, 
     "impactUnit" : 'Affected homes', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 1409, 
     "impactUnit" : 'Flooded homes', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 2, 
     "impactUnit" : 'Destroyed homes', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Informal settlements",
     "impactValue" : 13, 
     "impactUnit" : 'shelters', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }
]

In [71]:
i=8
print(f"{reports_to_label.iloc[i].appealCode}: {reports_to_label.iloc[i].date}")
print(reports_to_label.iloc[i].reportLink)
reports_to_label.iloc[i].nathaz_text

MDRYE011: 2023-05-29 00:00:00
https://go-api.ifrc.org/api/DownloadFile/67143/MDRYE011dfr


['SITUATION ANALYSIS Description of the disaster Heavy rains in Sanaa governorate caused extensive damage to public infrastructure, shelters for displaced people, and other private property.',
 'Three people died and two were injured due to the heavy rain.',
 'In Saadah Governorate, 299 families were affected, and approximately 50 of them were affected by heavy rains in various districts.',
 'Yemens annual rainy season starts in May and normally goes until August or September, but in 2022, Yemen witnessed heavier than normal rains, ranging in intensity, accompanied by thunders torms starting in May 2022.',
 'According to the Food and Agriculture Organization of the United Nations FAO Agrometeorological Early Warning Bulletin 1, the forecasts covering until July 31 favour the formation of heavy rains, especially affecting areas to the north of Ibb and central Hadramawt.',
 'Heavy rains and flooding caused significant damage in Yemen from May to mid September 2022, resulting in the l oss

In [22]:
labelled_impact_reports_dict["MDRYE011"]=[
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 76790, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ["P a g e 1 Internal DREF Operation n MDRYE011 Glide n FL-2022-000265-YEM Date of issue 29/07/2022 Expected timeframe 6 months Expected end date 31/01/2023 Category allocated to the of the disaster or crisis Orange DREF allocated CHF 452,156 Total number of people affected Approximately 76,790 people affected Number of people to be assisted 19,509 people (2,787 HH) Governorates affected Marib, Al Mahwit, Taiz, Ibb, Hadramawt, Al Bayda, Amran, Sadaa, Dhamar Al Hodeida Sana'a Hajjah, Al Mahra governorates Governorates targeted Al Hodeida, Hajjah Hadramout, and Al- Mahra, Marib and Sanaa Governorates Operating National Society Yemen Red Crescent Society has branches in all 22 Governates of Yemen, with 321 staff and 4,500 active volunteers, including 44 National Disaster Response trained team members, as well as trained first aid volunteers ready to deploy in case of emergency."],
     "country" : ["Yemen"],
     "location" : ["Marib", "Al Mahwit", "Taiz", "Ibb", "Hadramawt", "Al Bayda", "Amran", "Sadaa", "Dhamar Al Hodeida Sana'a Hajjah", "Al Mahra", "Al Hodeida", "Hajjah Hadramout", "Marib", "Sanaa"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : 2023,
     "endMonth" : 1,
     "endDay" : 21,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ["Situation analysis Description of the disaster On Saturday,23 July 2022, in Sana'a governorate, heavy rains led to floods causing extensive damage to public infrastructure, shelters for displaced people and other private property."],
     "country" : ["Yemen"],
     "location" : ["Sanaa governorate"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : 2023,
     "endMonth" : 1,
     "endDay" : 21,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ["Situation analysis Description of the disaster On Saturday,23 July 2022, in Sana'a governorate, heavy rains led to floods causing extensive damage to public infrastructure, shelters for displaced people and other private property."],
     "country" : ["Yemen"],
     "location" : ["Sanaa governorate"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : 2023,
     "endMonth" : 1,
     "endDay" : 21,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 3, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ["Situation analysis Description of the disaster On Saturday,23 July 2022, in Sana'a governorate, heavy rains led to floods causing extensive damage to public infrastructure, shelters for displaced people and other private property.", 
                     '3 people died and 2 people were injured due to the heavy rain that led to the collapse of their house which consist of three floors and 3 families.'],
     "country" : ["Yemen"],
     "location" : ["Sanaa governorate"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : 2023,
     "endMonth" : 1,
     "endDay" : 21,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Injured People",
     "impactValue" : 2, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ["Situation analysis Description of the disaster On Saturday,23 July 2022, in Sana'a governorate, heavy rains led to floods causing extensive damage to public infrastructure, shelters for displaced people and other private property.", 
                     '3 people died and 2 people were injured due to the heavy rain that led to the collapse of their house which consist of three floors and 3 families.'],
     "country" : ["Yemen"],
     "location" : ["Sanaa governorate"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : 2023,
     "endMonth" : 1,
     "endDay" : 21,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 56, 
     "impactUnit" : "families", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['the heavy rains affected 56 families in Al-Khamis camp, 137 families at IDPs camp in Al-Hasaba, 116 families at Aser camp in addition to 63 families at Al-Tahreer Square.'],
     "country" : ["Yemen"],
     "location" : ["Al-Khamis camp"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : 2023,
     "endMonth" : 1,
     "endDay" : 21,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 137, 
     "impactUnit" : "families", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['the heavy rains affected 56 families in Al-Khamis camp, 137 families at IDPs camp in Al-Hasaba, 116 families at Aser camp in addition to 63 families at Al-Tahreer Square.'],
     "country" : ["Yemen"],
     "location" : ["Al-Hasaba camp"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : 2023,
     "endMonth" : 1,
     "endDay" : 21,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 116, 
     "impactUnit" : "families", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['the heavy rains affected 56 families in Al-Khamis camp, 137 families at IDPs camp in Al-Hasaba, 116 families at Aser camp in addition to 63 families at Al-Tahreer Square.'],
     "country" : ["Yemen"],
     "location" : ["Aser camp"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : 2023,
     "endMonth" : 1,
     "endDay" : 21,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 63, 
     "impactUnit" : "families", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['the heavy rains affected 56 families in Al-Khamis camp, 137 families at IDPs camp in Al-Hasaba, 116 families at Aser camp in addition to 63 families at Al-Tahreer Square.'],
     "country" : ["Yemen"],
     "location" : ["Al-Tahreer Square"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : 2023,
     "endMonth" : 1,
     "endDay" : 21,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 299, 
     "impactUnit" : "families", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['In Saada Governorate,299 families were affected by heavy rains, Emergency Plan of Action (EPoA) Yemen Sanaa Floods P a g e 2 Internal and approximately 50 families of them were affected by heavy rains in the districts of Saada, Sahara, and Majaz.'],
     "country" : ["Yemen"],
     "location" : ["Saada Governorate"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : 2023,
     "endMonth" : 1,
     "endDay" : 21,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 50, 
     "impactUnit" : "families", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['In Saada Governorate,299 families were affected by heavy rains, Emergency Plan of Action (EPoA) Yemen Sanaa Floods P a g e 2 Internal and approximately 50 families of them were affected by heavy rains in the districts of Saada, Sahara, and Majaz.'],
     "country" : ["Yemen"],
     "location" : ["Saada district", "Sahara district", "Majaz district"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : 2023,
     "endMonth" : 1,
     "endDay" : 21,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 371, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Al-Mahra"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : 2023,
     "endMonth" : 7,
     "endDay" : 31,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 504, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Al-Mahwit"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : 2023,
     "endMonth" : 7,
     "endDay" : 31,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 1085, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Hadramout"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : 2023,
     "endMonth" : 7,
     "endDay" : 31,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 1127, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Dhamar"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : 2023,
     "endMonth" : 7,
     "endDay" : 31,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 1211, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Al-Bayda"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : 2023,
     "endMonth" : 7,
     "endDay" : 31,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 1239, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Saada"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : 2023,
     "endMonth" : 7,
     "endDay" : 31,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 1610, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Ibb"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : 2023,
     "endMonth" : 7,
     "endDay" : 31,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 2828, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Al- Hodeida"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : 2023,
     "endMonth" : 7,
     "endDay" : 31,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 3087, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Amran"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : 2023,
     "endMonth" : 7,
     "endDay" : 31,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 3290, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Al-Dhalea"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : 2023,
     "endMonth" : 7,
     "endDay" : 31,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 18039, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Marib"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : 2023,
     "endMonth" : 7,
     "endDay" : 31,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 19600, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Taiz"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : 2023,
     "endMonth" : 7,
     "endDay" : 31,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 20300, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Hajjah"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : 2023,
     "endMonth" : 7,
     "endDay" : 31,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 2499, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Sana’a"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : 2023,
     "endMonth" : 7,
     "endDay" : 31,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.', 
                     'Floods and storms have caused the total or partial destruction of tents, loss of personal belongings, food and essential household items, and damage to water tanks and sewage networks.'],
     "country" : ["Yemen"],
     "location" : ["Al-Mahra", "Al-Mahwit", "Hadramout", "Dhamar", "Al-Bayda", "Saada", "Ibb", "Al- Hodeida", "Amran", "Al-Dhalea", "Marib", "Taiz", "Hajjah", "Sana’a"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : 2023,
     "endMonth" : 7,
     "endDay" : 31,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Other Economic & Livelihood impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.', 
                     'Floods and storms have caused the total or partial destruction of tents, loss of personal belongings, food and essential household items, and damage to water tanks and sewage networks.'],
     "country" : ["Yemen"],
     "location" : ["Al-Mahra", "Al-Mahwit", "Hadramout", "Dhamar", "Al-Bayda", "Saada", "Ibb", "Al- Hodeida", "Amran", "Al-Dhalea", "Marib", "Taiz", "Hajjah", "Sana’a"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : 2023,
     "endMonth" : 7,
     "endDay" : 31,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Water, Sanitation, and Hygiene Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.', 
                     'Floods and storms have caused the total or partial destruction of tents, loss of personal belongings, food and essential household items, and damage to water tanks and sewage networks.'],
     "country" : ["Yemen"],
     "location" : ["Al-Mahra", "Al-Mahwit", "Hadramout", "Dhamar", "Al-Bayda", "Saada", "Ibb", "Al- Hodeida", "Amran", "Al-Dhalea", "Marib", "Taiz", "Hajjah", "Sana’a"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : 2023,
     "endMonth" : 7,
     "endDay" : 31,
     "hazards" : ["Flood", "Other storm"],
    }
]

In [76]:
df_impact_list = []
for k,v in labelled_impact_reports_dict.items():
    df_impact = pd.DataFrame(v)
    df_impact['appealCode'] = k
    df_impact_list.append(df_impact)
df_impact_all = pd.concat(df_impact_list)
df_impact_all.reset_index(inplace=True, drop=True)

#Save impact csv 
fn = f"labelled_reports_impact_laura_15-08-25.csv"
df_impact_all.to_csv(DATA_LABELLED+fn, index=False)

C:\Users\lhasbini\AppData\Local\Temp\ipykernel_8096\826530911.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_impact_all = pd.concat(df_impact_list)


In [75]:
df_impact_all.head()

,reportDate,impactSubtype,impactValue,impactUnit,impactValuePrecision,impactValueMin,impactValueMax,annotation,country,location,startYear,startMonth,startDay,endYear,endMonth,endDay,hazards,appealCode
0,2019-07-19,Affected People,2176519.0,people,exact,NaN,NaN,[DREF operation n MDRBD022 Glide n FL-2019-000...,[Bangladesh],None,2019.0,7.0,18.0,2019.0,11.0,18.0,[Flood],MDRBD022
1,2019-07-19,Affected People,NaN,people,approx,1000000.0,NaN,[While the monsoon season normally brings annu...,"[Nepal, India]",None,2019.0,NaN,NaN,NaN,NaN,NaN,[Flood],MDRBD022
2,2019-07-19,Affected People,NaN,people,approx,2100000.0,NaN,[According to National disaster response coord...,[Bangladesh],"[Kurigram district, Gaibandha district, Lalmon...",2019.0,7.0,16.0,NaN,NaN,NaN,"[Flood, Mass movement]",MDRBD022
3,2019-07-19,Residential Buildings,100000.0,houses destroyed,approx,NaN,NaN,[According to National disaster response coord...,[Bangladesh],"[Kurigram district, Gaibandha district, Lalmon...",2019.0,7.0,16.0,NaN,NaN,NaN,"[Flood, Mass movement]",MDRBD022
4,2019-07-19,Crop Production and Forestry,14733.0,hectares of crops,approx,NaN,NaN,[According to National disaster response coord...,[Bangladesh],"[Kurigram district, Gaibandha district, Lalmon...",2019.0,7.0,16.0,NaN,NaN,NaN,"[Flood, Mass movement]",MDRBD022


## OLD LABELLING

In [51]:
labelled_impact_reports_dict = {}

In [64]:
labelled_impact_reports_dict["MDRBD015"]=[
    {"reportDate": "2015-09-16",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The Government district level ‘D-form’ data immediately after the disaster indicated many houses were flattened or under water, trees uprooted, and power supplies and communication systems disrupted in some places.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Tropical storm"],
    },
    {"reportDate": "2015-09-16",
     "impactSubtype" : "IT and Communication Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The Government district level ‘D-form’ data immediately after the disaster indicated many houses were flattened or under water, trees uprooted, and power supplies and communication systems disrupted in some places.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Tropical storm"],
    },
    {"reportDate": "2015-09-16",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Crops were damaged and shrimp projects flooded.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Tropical storm"],
    },
    {"reportDate": "2015-09-16",
     "impactSubtype" : "Affected People",
     "impactValue" : 2600000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The cumulative effect of the floods coming after Cyclone Komen increased the affected population to 2.6 million people.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Tropical storm"],
    },
    {"reportDate": "2015-09-16",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['An aerial survey was conducted on 30 August 2015 in the northern districts to observe the flooding situation and the potential damage to housing, agriculture, and infrastructure and to map the scale of population displacement.'],
     "country" : ["Bangladesh"],
     "location" : ['northern districts'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Tropical storm"],
    },
    {"reportDate": "2015-09-16",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['An aerial survey was conducted on 30 August 2015 in the northern districts to observe the flooding situation and the potential damage to housing, agriculture, and infrastructure and to map the scale of population displacement.'],
     "country" : ["Bangladesh"],
     "location" : ['northern districts'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Tropical storm"],
    },
    {"reportDate": "2015-09-16",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['An aerial survey was conducted on 30 August 2015 in the northern districts to observe the flooding situation and the potential damage to housing, agriculture, and infrastructure and to map the scale of population displacement.'],
     "country" : ["Bangladesh"],
     "location" : ['northern districts'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Tropical storm"],
    },
    {"reportDate": "2015-09-16",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Floods have caused extensive damage to crops in different parts of the country.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    }
]

In [65]:
labelled_impact_reports_dict["MDRBD022"]=[
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 7600000, 
     "impactValueMax" : None, 
     "annotation" : ['According to the National Needs Assessment Working Group NAWG, Bangladesh situation report dated 28 July 2019, more than 7.6 million people were affected in 28 districts, over 300,000 people displaced, approximately 600,000 houses damaged, and 114 people dead.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 300000, 
     "impactValueMax" : None, 
     "annotation" : ['According to the National Needs Assessment Working Group NAWG, Bangladesh situation report dated 28 July 2019, more than 7.6 million people were affected in 28 districts, over 300,000 people displaced, approximately 600,000 houses damaged, and 114 people dead.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "locationAnnotation" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 600000, 
     "impactUnit" : "houses", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['According to the National Needs Assessment Working Group NAWG, Bangladesh situation report dated 28 July 2019, more than 7.6 million people were affected in 28 districts, over 300,000 people displaced, approximately 600,000 houses damaged, and 114 people dead.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 114, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['According to the National Needs Assessment Working Group NAWG, Bangladesh situation report dated 28 July 2019, more than 7.6 million people were affected in 28 districts, over 300,000 people displaced, approximately 600,000 houses damaged, and 114 people dead.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : 532000, 
     "impactUnit" : "hectares of crops", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['On top of that, according to the media, about 532,000 hectares of crops were destroyed.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Other Infrastructural impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['It was also reported that embankments were damaged and inundated.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Affected People",
     "impactValue" : 40000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Due to this second spell of flood, around 40,000 people were affected according to the NAWG.', 
                     'In addition, Bangladesh experienced another spell of flood in northeastern districts named Rajshahi, Shariatpur, Kushtia, Rajbari, Chapai Nawabganj, Pabna and Natore during the first week of October 2019, which affected some new areas.'],
     "country" : ["Bangladesh"],
     "location" : ["Rajshahi", "Shariatpur", "Kushtia", "Rajbari", "Chapai Nawabganj", "Pabna", "Natore"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Healthcare Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['While more than half of the country experiencing the sufferings from monsoon floods, the Dengue situation in the country reached peak transmission during June to October 2019, with hospitals overflowing with patients and the rising number of dengue patients had broken all previous records.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : 2019,
     "startMonth" : 6,
     "startDay" : None,
     "endYear" : 2019,
     "endMonth" : 10,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Ill People",
     "impactValue" : 148, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The Institute of Epidemiology, Disease Control and Research IEDCR of Government of Bangladesh GoB had received 266 reports of denguerelated deaths and after reviewing it, confirmed 148 deaths till December 2019.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2019,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Epidemic"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['During December 2019 and January 2020, the country experienced several cold waves over different districts of Chuadanga, Dinajpur, Panchagarh, Rajshahi, Pabna, Naogaon, Nilphamari, Jessore, Bogura, Lalmonirhat, Gaibandha, Kurigram, Sirajganj, Tangail, Jamalpur, etc., disrupting normal life and causing suffering to the people.'],
     "country" : ["Bangladesh"],
     "location" : ["Chuadanga", "Dinajpur", "Panchagarh", "Rajshahi", "Pabna", "Naogaon", "Nilphamari", "Jessore", "Bogura", "Lalmonirhat", "Gaibandha", "Kurigram", "Sirajganj", "Tangail", "Jamalpur"],
     "startYear" : 2019,
     "startMonth" : 12,
     "startDay" : None,
     "endYear" : 2020,
     "endMonth" : 1,
     "endDay" : None,
     "hazards" : ["Extreme cold temperature"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : "people",
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 2400000, 
     "impactValueMax" : None,  
     "annotation" : ['Following the great danger signal and evacuation order of the GoB, more than 2.4 million people were moved to 14,636 permanent and temporary shelters in 19 coastal districts before the cyclone hit the countrys coast.'],
     "country" : ["Bangladesh"],
     "location" : ['coastal districts'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Affected People",
     "impactValue" : 2600000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Approximately 2.6 million people were affected 205,368 houses were damaged 55,767 houses were destroyed in the 19 affected districts.'],
     "country" : ["Bangladesh"],
     "location" : ["19 affected districts"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 205368, 
     "impactUnit" : "houses damaged", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Approximately 2.6 million people were affected 205,368 houses were damaged 55,767 houses were destroyed in the 19 affected districts.'],
     "country" : ["Bangladesh"],
     "location" : ["19 affected districts"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 55767, 
     "impactUnit" : "houses destroyed", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Approximately 2.6 million people were affected 205,368 houses were damaged 55,767 houses were destroyed in the 19 affected districts.'],
     "country" : ["Bangladesh"],
     "location" : ["19 affected districts"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Water, Sanitation, and Hygiene Infrastructure",
     "impactValue" : 40894, 
     "impactUnit" : "latrines", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['In addition to that 40,894 latrines 18,235 water points 32,037 hectares of crops and vegetable 18,707 hectares of fish cultivation area 440 kilometers of road and 76 kilometers of embankment were damaged.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Water, Sanitation, and Hygiene Infrastructure",
     "impactValue" : 18235, 
     "impactUnit" : "water points", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['In addition to that 40,894 latrines 18,235 water points 32,037 hectares of crops and vegetable 18,707 hectares of fish cultivation area 440 kilometers of road and 76 kilometers of embankment were damaged.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Agriculture",
     "impactValue" : 32037, 
     "impactUnit" : "hectares of crops and vegetable", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['In addition to that 40,894 latrines 18,235 water points 32,037 hectares of crops and vegetable 18,707 hectares of fish cultivation area 440 kilometers of road and 76 kilometers of embankment were damaged.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Affected Livestock and Animals",
     "impactValue" : 18707, 
     "impactUnit" : "hectares of fish cultivation", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['In addition to that 40,894 latrines 18,235 water points 32,037 hectares of crops and vegetable 18,707 hectares of fish cultivation area 440 kilometers of road and 76 kilometers of embankment were damaged.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Road Infrastructure",
     "impactValue" : 440, 
     "impactUnit" : "km roads", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['In addition to that 40,894 latrines 18,235 water points 32,037 hectares of crops and vegetable 18,707 hectares of fish cultivation area 440 kilometers of road and 76 kilometers of embankment were damaged.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Other Infrastructural impact",
     "impactValue" : 76, 
     "impactUnit" : "km embankment", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['In addition to that 40,894 latrines 18,235 water points 32,037 hectares of crops and vegetable 18,707 hectares of fish cultivation area 440 kilometers of road and 76 kilometers of embankment were damaged.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Affected People",
     "impactValue" : 5400000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['On the other hand, severe floods that struck Bangladesh during the last week of June 2020, driven by heavy monsoon and upstream water, have prolonged and intensified suffering of 5.4 million people in the northern, central and north eastern part of the country.'],
     "country" : ["Bangladesh"],
     "location" : ["northern part", "central part", "north eastern part"],
     "startYear" : 2020,
     "startMonth" : 6,
     "startDay" : None,
     "endYear" : 2020,
     "endMonth" : 6,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Till the beginning of October 2020, due to monsoon raining and heavy rainfall in upstream, people in many districts suffered with different spells of floods.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2020,
     "endMonth" : 10,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2020-12-05",
     "impactSubtype" : "Water, Sanitation, and Hygiene Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['There have been widespread damages in livelihoods shelter water, sanitation and hygiene WASH sectors in most of the affected districts.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }
]

In [66]:
labelled_impact_reports_dict["MDRCN006"]=[
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Affected People",
     "impactValue" : 1381000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Sichuan", "southeast region of Gansu"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 3, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Sichuan", "southeast region of Gansu"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Displaced People",
     "impactValue" : 222000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Sichuan", "southeast region of Gansu"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Injured People",
     "impactValue" : 22000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Sichuan", "southeast region of Gansu"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : "houses collapsed", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 900, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Deyang", "Mianyang", "Guangyuan"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : "houses damaged", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 29000, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Deyang", "Mianyang", "Guangyuan"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : 36900, 
     "impactUnit" : "hectares of crops", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['A total of 36,900 hectares of crops were also affected by the flood.'],
     "country" : ["China"],
     "location" : ["Sichuan", "southeast region of Gansu"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : None, 
     "impactUnit" : "CHF", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 792000000, 
     "impactValueMax" : None, 
     "annotation" : ['The direct economic loss was estimated to be over 5.3 billion Yuan approximately CHF 792 million.'],
     "country" : ["China"],
     "location" : ["Sichuan", "southeast region of Gansu"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Affected People",
     "impactValue" : 1519000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 12, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Missing People",
     "impactValue" : 4, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Displaced People",
     "impactValue" : 30000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : "houses collapsed", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 2300, 
     "impactValueMax" : None, 
     "annotation" : ['More than 2,300 houses collapsed, and 19,000 were damaged to varying degrees.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : "houses damaged", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 19000, 
     "impactValueMax" : None, 
     "annotation" : ['More than 2,300 houses collapsed, and 19,000 were damaged to varying degrees.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 538000000, 
     "impactUnit" : "CHF", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The direct economic loss was estimated 3.6 billion Yuan approximately CHF 538 million.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : "houses", 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['According to this rapid assessment, heavy rainfall had resulted in a large number of seriously damaged houses that have continued to collapse in these two provinces.'],
     "country" : ["China"],
     "location" : ["Sichuan", "southeast region of Gansu"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }
]

In [68]:
labelled_impact_reports_dict["MDRMY003"]=[
    {"reportDate": "2017-11-21",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 23000, 
     "impactValueMax" : None, 
     "annotation" : ['More than 23,000 people, mainly from smaller towns and villages in rural areas, had to leave their homes to established relief centres.', 
                     'Situation analysis Description of the disaster Heavy rains that started in December 2016 continued until late January 2017 in parts of Malaysia, causing flooding in seven states of Peninsular Malaysia Johor, Kelantan, Pahang, Perak, Terengganu, Malacca and Selangor and Sabah in East Malaysia.'],
     "country" : ["Malaysia"],
     "location" : ["Johor", "Kelantan", "Pahang", "Perak", "Terengganu", "Malacca and Selangor" and "Sabah"],
     "startYear" : 2016,
     "startMonth" : 12,
     "startDay" : None,
     "endYear" : 2017,
     "endMonth" : 1,
     "endDay" : 31,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2017-11-21",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 8000, 
     "impactValueMax" : None, 
     "annotation" : ['National Agency for Disaster Administration (NADMA) reported that the state of Johor suffered the brunt of rising waters, with more than 8,000 evacuees and a fatality.'],
     "country" : ["Malaysia"],
     "location" : ["Johor"],
     "startYear" : 2016,
     "startMonth" : 12,
     "startDay" : None,
     "endYear" : 2017,
     "endMonth" : 1,
     "endDay" : 31,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2017-11-21",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 1, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['National Agency for Disaster Administration (NADMA) reported that the state of Johor suffered the brunt of rising waters, with more than 8,000 evacuees and a fatality.'],
     "country" : ["Malaysia"],
     "location" : ["Johor"],
     "startYear" : 2016,
     "startMonth" : 12,
     "startDay" : None,
     "endYear" : 2017,
     "endMonth" : 1,
     "endDay" : 31,
     "hazards" : ["Flood"]
    }
]

In [69]:
df_impact_list = []
for k,v in labelled_impact_reports_dict.items():
    df_impact = pd.DataFrame(v)
    df_impact['appealCode'] = k
    df_impact_list.append(df_impact)
df_impact_all = pd.concat(df_impact_list)
df_impact_all.reset_index(inplace=True, drop=True)

#Save impact csv 
fn = f"labelled_old_{i+1}reports_impact_laura.csv"
df_impact_all.to_csv(DATA_LABELLED+fn, index=False)

In [70]:
df_impact_all

,reportDate,impactSubtype,impactValue,impactUnit,impactValuePrecision,impactValueMin,impactValueMax,annotation,country,location,startYear,startMonth,startDay,endYear,endMonth,endDay,hazards,appealCode,locationAnnotation
0,2015-09-16,Residential Buildings,NaN,None,None,NaN,None,[The Government district level ‘D-form’ data i...,[Bangladesh],None,NaN,NaN,NaN,NaN,NaN,NaN,"[Flood, Tropical storm]",MDRBD015,NaN
1,2015-09-16,IT and Communication Infrastructure,NaN,None,None,NaN,None,[The Government district level ‘D-form’ data i...,[Bangladesh],None,NaN,NaN,NaN,NaN,NaN,NaN,"[Flood, Tropical storm]",MDRBD015,NaN
2,2015-09-16,Crop Production and Forestry,NaN,None,None,NaN,None,[Crops were damaged and shrimp projects flooded.],[Bangladesh],None,NaN,NaN,NaN,NaN,NaN,NaN,"[Flood, Tropical storm]",MDRBD015,NaN
3,2015-09-16,Affected People,2600000.0,people,exact,NaN,None,[The cumulative effect of the floods coming af...,[Bangladesh],None,NaN,NaN,NaN,NaN,NaN,NaN,"[Flood, Tropical storm]",MDRBD015,NaN
4,2015-09-16,Residential Buildings,NaN,None,None,NaN,None,[An aerial survey was conducted on 30 August 2...,[Bangladesh],[northern districts],NaN,NaN,NaN,NaN,NaN,NaN,"[Flood, Tropical storm]",MDRBD015,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,2023-11-30,"Water, Sanitation, and Hygiene Infrastructure",230.0,latrines,exact,NaN,None,"[Some 1213 households that is 8,491 people are...",[Guinea],"[Conakry, Ratoma, Matoto, Conakry Wareah Kinif...",2023.0,8.0,11.0,NaN,NaN,NaN,[Flood],MDRGN015,NaN
75,2023-11-30,"Water, Sanitation, and Hygiene Infrastructure",28.0,water points,exact,NaN,None,"[Some 1213 households that is 8,491 people are...",[Guinea],"[Conakry, Ratoma, Matoto, Conakry Wareah Kinif...",2023.0,8.0,11.0,NaN,NaN,NaN,[Flood],MDRGN015,NaN
76,2017-11-21,Displaced People,NaN,people,approx,23000.0,None,"[More than 23,000 people, mainly from smaller ...",[Malaysia],"[Johor, Kelantan, Pahang, Perak, Terengganu, S...",2016.0,12.0,NaN,2017.0,1.0,31.0,[Flood],MDRMY003,NaN
77,2017-11-21,Displaced People,NaN,people,approx,8000.0,None,[National Agency for Disaster Administration (...,[Malaysia],[Johor],2016.0,12.0,NaN,2017.0,1.0,31.0,[Flood],MDRMY003,NaN


In [16]:
i=2
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRBZ006: 2021-08-19 00:00:00


['The remaining balance of 131,747 CHF will be returned to the Disaster Relief Emergency Fund.',
 'The major donors and partners of the Disaster Relief Emergency Fund DREF include the Red Cross Societies and governments of Belgium, Britain, Canada, Denmark, German, Ireland, Italy, Japan, Luxembourg, New Zealand, Norway, Republic of Korea, Spain, Sweden and Switzerland, as well as DG ECHO, Blizzard Entertainment, Mondelez International Foundation, Fortive Corporation and other corporate and private donors.',
 'The IFRC, on behalf of the CRRC, would like to extend thanks to all for their generous contributions.',
 'Click here for the final financial report and here for the contact information.',
 'A.',
 'SITUATION ANALYSIS Description of the disaster Hurricane Eta made landfall on Nicaraguas shores as a strong Category 4 hurricane on November 4, 2020, causing destruction and excessive rain with a wind speed of 140 mph.',
 'Several Central American countries experienced the negative effec

In [17]:
labelled_impact_reports_dict["MDRBZ006"]=[
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['More than 40 communities were affected, mainly along the Mopan, Macal, Belize, and Sibun rivers.'],
     "country" : "Belize",
     "location" : ['Mopan river'],
     "locationAnnotation" : ['More than 40 communities were affected, mainly along the Mopan, Macal, Belize, and Sibun rivers.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Storm"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['More than 40 communities were affected, mainly along the Mopan, Macal, Belize, and Sibun rivers.'],
     "country" : "Belize",
     "location" : ['Macal river'],
     "locationAnnotation" : ['More than 40 communities were affected, mainly along the Mopan, Macal, Belize, and Sibun rivers.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Storm"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['More than 40 communities were affected, mainly along the Mopan, Macal, Belize, and Sibun rivers.'],
     "country" : "Belize",
     "location" : ['Belize river'],
     "locationAnnotation" : ['More than 40 communities were affected, mainly along the Mopan, Macal, Belize, and Sibun rivers.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Storm"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['More than 40 communities were affected, mainly along the Mopan, Macal, Belize, and Sibun rivers.'],
     "country" : "Belize",
     "location" : ['Sibun river'],
     "locationAnnotation" : ['More than 40 communities were affected, mainly along the Mopan, Macal, Belize, and Sibun rivers.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Storm"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : 60000, 
     "annotation" : ['According to the National Emergency Management Agency DREF Operation Final Report Belize Hurricane Eta P a g e 2 NEMO, 60,000 people living along the impacted areas have been affected, while some 5,000 persons were directly impacted1.'],
     "country" : "Belize",
     "location" : None,
     "locationAnnotation" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Storm"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "annotation" : ['The floods caused damage to residential property, utilities, farms, and road infrastructure and further added to the vulnerabilities due to COVID19, which has had a strong effect on the tourism industry, leaving many families with limited income.'],
     "country" : "Belize",
     "location" : None,
     "locationAnnotation" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Transportation Infrastructure",
     "impactValue" : None, 
     "annotation" : ['The floods caused damage to residential property, utilities, farms, and road infrastructure and further added to the vulnerabilities due to COVID19, which has had a strong effect on the tourism industry, leaving many families with limited income.'],
     "country" : "Belize",
     "location" : None,
     "locationAnnotation" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "country" : "Belize",
     "location" : ["Arenal"],
     "locationAnnotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "country" : "Belize",
     "location" : ["Benque"],
     "locationAnnotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "country" : "Belize",
     "location" : ["Calla Creek"],
     "locationAnnotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "country" : "Belize",
     "location" : ["Bullet Tree Falls"],
     "locationAnnotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "country" : "Belize",
     "location" : ["Valley of Peace"],
     "locationAnnotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "country" : "Belize",
     "location" : ["Santa Familia"],
     "locationAnnotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "country" : "Belize",
     "location" : ["Blackman Eddy"],
     "locationAnnotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "country" : "Belize",
     "location" : ["Roaring Creek"],
     "locationAnnotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "country" : "Belize",
     "location" : ["La Rivera"],
     "locationAnnotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "country" : "Belize",
     "location" : ["Bomba"],
     "locationAnnotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "country" : "Belize",
     "location" : ["Maskall"],
     "locationAnnotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "country" : "Belize",
     "location" : ["Crooked Tree"],
     "locationAnnotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "country" : "Belize",
     "location" : ["May Pen"],
     "locationAnnotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "country" : "Belize",
     "location" : ["Rancho Dolores"],
     "locationAnnotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "country" : "Belize",
     "location" : ["Freetown Sibun"],
     "locationAnnotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2021-03-25",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "annotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "country" : "Belize",
     "location" : ["Lemonal"],
     "locationAnnotation" : ['The hardest hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
]

In [18]:
labelled_reports_dict['MDRBZ006']=[
    {"reportDate": "2021-03-25",
     "hazards": 'Storm',
     "hazardSubtypes" : ["tropical storm"],
     "country" : 'Nicaragua',
     "location" : None,
     "locationAnnotation" : ['SITUATION ANALYSIS Description of the disaster Hurricane Eta made landfall on Nicaragua’s shores as a strong Category 4 hurricane on 4 November 2020, causing destruction and excessive rain with a wind speed of 140 mph.'],
     "startYear" : 2020,
     "startMonth" : 11,
     "startDay" : 4,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : 'Eta',
    },
    {"reportDate": "2021-03-25",
     "hazards": 'Storm',
     "hazardSubtypes" : ["tropical storm"],
     "country" : 'Belize',
     "location" : None,
     "locationAnnotation" : ['Several Central American countries experienced the negative effects of Hurricane Eta, including Belize.'],
     "startYear" : 2020,
     "startMonth" : 11,
     "startDay" : 3,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : 'Eta',
    },
    {"reportDate": "2021-03-25",
     "hazards": 'Flood',
     "hazardSubtypes" : None,
     "country" : 'Belize',
     "location" : ["Western District of Cayo",
                 "Belize District",
                 "Mopan river",
                 "Macal river",
                 "Belize river",
                 "Sibun river"
                 "Cayo District",
                   "Belize City",
                   "Arenal",
                   "Roaring Creek"],
     "locationAnnotation" :  ['Approximately twenty inches of rainfall, caused severe flooding in the Western District of Cayo and Belize District, including Belize City.',
                              "More than 40 communities were affected, mainly along the Mopan, Macal, Belize, and Sibun rivers."
                              "In the Cayo District, the Macal and Mopan rivers rose more than 8.8 meters, inundating every village from Arenal to Roaring Creek."],
     "startYear" : 2020,
     "startMonth" : 11,
     "startDay" : 3,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"reportDate": "2021-03-25",
     "hazards": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Nicaragua",
     "region" : None,
     "city" : None,
     "locationAnnotation" : ["Additionally, on 16 November, Hurricane Iota made landfall in Nicaragua, which brought additional rain to Belize and exacerbated the floods in many areas."],
     "startYear" : 2020,
     "startMonth" : 11,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Iota",
    },
    {"reportDate": "2021-03-25",
     "hazards": "Flood",
     "hazardSubtypes" : None,
     "country" : "Belize",
     "location" : ["Arenal",
               "Benque",
               "Calla Creek",
               "Bullet Tree Falls",
               "Valley of Peace",
               "Santa Familia",
               "Blackman Eddy",
               "Roaring Creek",
               "La Rivera",
               "Bomba",
               "Maskall",
               "Crooked Tree",
               "May Pen",
               "Rancho Dolores",
               "Freetown Sibun",
               "Lemonal"],
     "locationAnnotation" : ["Additionally, on 16 November, Hurricane Iota made landfall in Nicaragua, which brought additional rain to Belize and exacerbated the floods in many areas.",
                             "Among the hardest-hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal."],
     "startYear" : 2020,
     "startMonth" : 11,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Iota",
    },
    #{"hazards": 'Flood',
    # "hazardSubtypes" : None,
    # "country" : 'Belize',
    # "region" : "Belize District",
    # "city" : None,
    # "locationAnnotation" :  'Approximately twenty inches of rainfall, caused severe flooding in the Western District of Cayo and Belize District, including Belize City.',
    # "startYear" : 2020,
    # "startMonth" : 11,
    # "startDay" : 3,
    # "endYear" : None,
    # "endMonth" : None,
    # "endDay" : None,
    # "hazardName" : None,
    #},
    #{"hazards": 'Flood',
    # "hazardSubtypes" : None,
    # "country" : 'Belize',
    # "region" : "Belize District",
    # "city" : "Belize City",
    # "locationAnnotation" :  'Approximately twenty inches of rainfall, caused severe flooding in the Western District of Cayo and Belize District, including Belize City.',
    # "startYear" : 2020,
    # "startMonth" : 11,
    # "startDay" : 3,
    # "endYear" : None,
    # "endMonth" : None,
    # "endDay" : None,
    # "hazardName" : None,
    #},
]

In [19]:
i=3
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRCN006: 2019-03-14 00:00:00


['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.',
 'In some areas of North Central Sichuan, there were heavy rainstorms and torrential rains for four consecutive days.',
 'These were also compounded by the effects of two weather systems in the area Typhoon Prapiroon, and Typhoon Maria.',
 'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.',
 'A total of 36,900 hectares of crops were also affected by the flood.',
 'The direct economic loss was estimated to be over 5.3 billion Yuan approximately CHF 792 million.',
 'Gansu province was hit even h

In [20]:
labelled_impact_reports_dict["MDRCN006"]=[
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Affected People",
     "impactValue" : 1381000, 
     "annotation" : ['According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : "China",
     "location" : ["Sichuan", "Deyang", "Mianyang", "Guangyuan"],
     "locationAnnotation" : ['According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 13,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ['Flood', 'Storm'],
    },
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 3, 
     "annotation" : ['According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : "China",
     "location" : ["Sichuan", "Deyang", "Mianyang", "Guangyuan"],
     "locationAnnotation" : ['According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 13,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ['Flood', 'Storm'],
    },
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Displaced People",
     "impactValue" : 222000, 
     "annotation" : ['According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : "China",
     "location" : ["Sichuan", "Deyang", "Mianyang", "Guangyuan"],
     "locationAnnotation" : ['According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 13,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ['Flood', 'Storm'],
    },
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 29000, 
     "annotation" : ['According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : "China",
     "location" : ["Sichuan", "Deyang", "Mianyang", "Guangyuan"],
     "locationAnnotation" : ['According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 13,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ['Flood', 'Storm'],
    },
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Affected People",
     "impactValue" : 1519000, 
     "annotation" : ['The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "country" : "China",
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "locationAnnotation" : ['The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ['Flood'],
    },
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 12, 
     "annotation" : ['The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "country" : "China",
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "locationAnnotation" : ['The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ['Flood'],
    },
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Missing People",
     "impactValue" : 4, 
     "annotation" : ['The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "country" : "China",
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "locationAnnotation" : ['The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ['Flood'],
    },
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Displaced People",
     "impactValue" : 30000, 
     "annotation" : ['The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "country" : "China",
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "locationAnnotation" : ['The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ['Flood'],
    },
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "annotation" : ['More than 2,300 houses collapsed, and 19,000 were damaged to varying degrees.'],
     "country" : "China",
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "locationAnnotation" : ['The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ['Flood'],
    },
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "annotation" : ['According to this rapid assessment, heavy rainfall had resulted in a large number of seriously damaged houses that have continued to collapse in these two provinces.'],
     "country" : "China",
     "location" : ["Sichuan", "Gansu"],
     "locationAnnotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ['Flood'],
    },
]

In [21]:
labelled_reports_dict["MDRCN006"]=[
    {"reportDate": "2019-03-14",
     "hazards": "Flood",
     "hazardSubtypes" : None,
     "country" : "China",
     "location" : ["Sichuan",
                   "southeast region of Gansu Province",
                   "North Central Sichuan",
                   "Gansu province", 
                   "Deyang",
                   "Mianyang",
                   "Guangyuan",
                   "Tianshui",
                   "Zhangye",
                   "Pingliang"],
     "locationAnnotation" : ["SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.",
                             "In some areas of North Central Sichuan, there were heavy rainstorms and torrential rains for four consecutive days.",
                             "According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died; 222,000 had taken emergency resettlement; 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan (that includes 15 cities and 70 counties); more than 900 houses collapsed, and 29,000 houses damaged.",
                             "Gansu province was hit even harder, according to the Ministry of Emergency Management.",
                             "The area of Tianshui, Zhangye, Pingliang (including 10 cities and 46 counties) were flooded, and affected 1,519,000 people where 12 died; 4 missing; and 30,000 were evacuated.",
                             "The flooding season was rightly anticipated to continue until the end of August 2018 and more rain fall events were registered."],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : 2018,
     "endMonth" : 8,
     "endDay" : 31,
     "hazardName" : None,
    },
     {"reportDate": "2019-03-14",
     "hazards": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "China",
     "location" : ["Sichuan",
                 "southeast region of Gansu Province",
                 "North Central Sichuan"],
     "locationAnnotation" : ['These were also compounded by the effects of two weather systems in the area; Typhoon Prapiroon, and Typhoon Maria.',
                             ],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Typhoon Prapiroon",
    },
     {"reportDate": "2019-03-14",
     "hazards": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "China",
     "location" : ["Sichuan",
                 "southeast region of Gansu Province",
                 "North Central Sichuan"],
     "locationAnnotation" : ['These were also compounded by the effects of two weather systems in the area; Typhoon Prapiroon, and Typhoon Maria.',
                             ],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Typhoon Maria",
    },
    {"reportDate": "2019-03-14",
     "hazards": "Storm",
     "hazardSubtypes" : None,
     "country" : "China",
     "location" : ["Southeast Gansu"],
     "locationAnnotation" : ['A heavy rainstorm occurred in Southeast Gansu from 10 to 11 July 2018.',
                             ],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 10,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 11,
     "hazardName" : None,
    },

]

In [22]:
i=4
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRGN015: 2023-11-30 00:00:00


['Page 1 19 DREF Operational Update Guinea Floods Appeal MDRGN015 Total DREF Allocation Crisis Category Yellow Hazard Flood Glide Number FL2023000158GIN People Affected 24,135 people People Targeted 14,350 people Event Onset Sudden Operation Start Date 20230823 New Operational End Date 20240131 Total Operating Timeframe 5 months Additional Allocation Requested Targeted Areas Kindia Page 2 19 Description of the Event What happened, where and when?',
 'Guinea has been experiencing persistent torrential rains since the beginning of August 2023.',
 'The highest recorded incidents were on Sunday 6 August 2022 in Coyah, and on Friday 11 August 2023 in Conakry and Siguiri, with rains causing associated impacts, including flooding in lowlying areas as well as the overflow of rivers.',
 'Major roads in Conakry and Siguiri were rendered impassable due to the flood waters, heavily constraining vehicles, and pedestrians having to find alternative routes.',
 'Different prefectures across the countr

In [23]:
labelled_impact_reports_dict["MDRGN015"]=[
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Affected People",
     "impactValue" : 24135, 
     "annotation" : ['Page 1 / 19 DREF Operational Update Guinea Floods Appeal: MDRGN015 Total DREF Allocation: - Crisis Category: Yellow Hazard: Flood Glide Number: FL-2023-000158-GIN People Affected: 24,135 people People Targeted: 14,350 people Event Onset: Sudden Operation Start Date: 2023-08-23 New Operational End Date: 2024-01-31 Total Operating Timeframe: 5 months Additional Allocation Requested: - Targeted Areas: Kindia Page 2 / 19 Description of the Event What happened, where and when?'],
     "country" : "Guinea",
     "location" : None,
     "locationAnnotation" : None,
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 23,
     "endYear" : 2024,
     "endMonth" : 1,
     "endDay" : 31,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Transportation Infrastructure",
     "impactValue" : None, 
     "annotation" : ['Major roads in Conakry and Siguiri were rendered impassable due to the flood waters, heavily constraining vehicles, and pedestrians having to find alternative routes.'],
     "country" : "Guinea",
     "location" : None,
     "locationAnnotation" : None,
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 23,
     "endYear" : 2024,
     "endMonth" : 1,
     "endDay" : 31,
     "hazards" : ["Flood"],
    },
]

In [24]:
labelled_reports_dict["MDRGN015"]=[
    {"reportDate": "2023-11-30",
     "hazards": "Flood",
     "hazardSubtypes" : ["riverine flood", "flash flood"],
     "country" : "Guinea",
     "location" : ["Kindia", 
                   "Coyah",
                   "Conakry",
                   "Siguiri"],
     "locationAnnotation" : ["Page 1 / 19 DREF Operational Update Guinea Floods Appeal: MDRGN015 Total DREF Allocation: - Crisis Category: Yellow Hazard: Flood Glide Number: FL-2023-000158-GIN People Affected: 24,135 people People Targeted: 14,350 people Event Onset: Sudden Operation Start Date: 2023-08-23 New Operational End Date: 2024-01-31 Total Operating Timeframe: 5 months Additional Allocation Requested: - Targeted Areas: Kindia Page 2 / 19 Description of the Event What happened, where and when?",
                             "The highest recorded incidents were on Sunday 6 August 2022 in Coyah, and on Friday 11 August 2023 in Conakry and Siguiri, with rains causing associated impacts, including flooding in low-lying areas as well as the overflow of rivers.",
                             "Major roads in Conakry and Siguiri were rendered impassable due to the flood waters, heavily constraining vehicles, and pedestrians having to find alternative routes.",],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 1,
     "endYear" : 2023,
     "endMonth" : 8,
     "endDay" : 11,
     "hazardName" : None,
    },

]

In [64]:
i=5
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRIQ014: 2023-05-12 00:00:00


['The major donors and partners of the Disaster Relief Emergency Fund (DREF) include the Red Cross Societies and governments of Belgium, Britain, Canada, Denmark, German, Ireland, Italy, Japan, Luxembourg, New Zealand, Norway, the Republic of Korea, Spain, Sweden, and Switzerland, as well as DG ECHO and Blizzard Entertainment, Mondelez International Foundation, and Fortive Corporation and other corporate and private donors.',
 'The IFRC, on behalf of the Iraqi Red Crescent Society, would like to extend thanks to all for their generous contributions.',
 'SITUATION ANALYSIS Description of the disaster Iraq is at risk of multiple disasters ranging from natural phenomena such as drought, sandstorms, heatwaves, and floods, to man-made ones.',
 'After one of the driest years in decades, heavy rains slammed Iraq’s northern Kurdish region on 17 December 2021.',
 'The overnight rainfall caused a flash flood in Erbil, the region’s capital, and the Kirkuk governorate in northern Iraq.',
 'Destruc

In [65]:
#empty dict structure to store results
labelled_reports_dict["MDRIQ014"]=[
    {"reportDate": "2023-05-12",
     "hazards": "Flood",
     "hazardSubtypes" : ["flash flood"],
     "country" : "Iraq",
     "region" : ["northern Kurdish",
                 "Erbil",
                 "Kirkuk",
                 "northern Iraq"],
     "city" : ["Daratu",
               "Qushtapa",
               "Shamamk",
               "Zhyan",
               "Roshinbiri",
               "Bahrka"],
     "locationAnnotation" : ["After one of the driest years in decades, heavy rains slammed Iraq’s northern Kurdish region on 17 December 2021.",
                             'The overnight rainfall caused a flash flood in Erbil, the region’s capital, and the Kirkuk governorate in northern Iraq.',
                              "In the early hours of the morning, muddy water inundated people’s homes in Erbil's Daratu, Qushtapa, Shamamk, Zhyan, Roshinbiri, and Bahrka neighborhoods, forcing inhabitants out of their houses.",
],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 17,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"reportDate": "2023-05-12",
     "hazards": "Drought",
     "hazardSubtypes" : ["drought"],
     "country" : "Iraq",
     "region" : None,
     "city" : None,
     "locationAnnotation" : ["After one of the driest years in decades, heavy rains slammed Iraq’s northern Kurdish region on 17 December 2021.",
                             ],
     "startYear" : 2021,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
]

In [66]:
i=6
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRLA009: 2024-05-31 00:00:00


['(Map: IFRC, IM) Date when the trigger was met 09-08-2023 What happened, where and when?',
 'During August 2023, persistent heavy rain led to extensive flooding across the central and southern regions of Laos.',
 'The flooding caused damages to numerous farms and houses, affecting thousands of people in the inundated areas.',
 'In the report released on 21 August 2023, the National Disaster Management Committee (NDMC), under the Ministry of Labour and Social Welfare (MOLSW), mentioned that 12 provinces were affected by the floods, including Vientiane Capital, Bokeo, Houaphan, Luang Prabang, Xaignabouli, Xiangkhouang, Vientiane, Bolikhamxai, Khammouan, Savannakhet, Champasak and Xaixomboun.',
 'The impact of the flooding was substantial, with a geographical scope that covered 550 villages across 50 districts within the 12 provinces.',
 'The agriculture sector was heavily affected by the floods, with massive damages to crops, cropland and fishponds, which put households in crisis as the

In [67]:
labelled_reports_dict["MDRLA009"]=[
    {"reportDate": "2024-05-31",
     "hazards": "Flood",
     "hazardSubtypes" : None,
     "country" : "Laos",
     "region" : ["Central region",
                 "Souther region",
                 "Vientiane Capital",
                  "Bokeo",
                  "Houaphan",
                  "Luang Prabang",
                  "Xaignabouli",
                  "Xiangkhouang",
                  "Vientiane",
                  "Bolikhamxai",
                  "Khammouan",
                  "Savannakhet",
                  "Champasak",
                  "Xaixomboun"],
     "city" : None,
     "locationAnnotation" : ["During August 2023, persistent heavy rain led to extensive flooding across the central and southern regions of Laos.",
                             "In the report released on 21 August 2023, the National Disaster Management Committee (NDMC), under the Ministry of Labour and Social Welfare (MOLSW), mentioned that 12 provinces were affected by the floods, including Vientiane Capital, Bokeo, Houaphan, Luang Prabang, Xaignabouli, Xiangkhouang, Vientiane, Bolikhamxai, Khammouan, Savannakhet, Champasak and Xaixomboun.",
                             ],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },

]

In [68]:
i=7
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRMG020: 2024-04-04 00:00:00


['Page 2 / 18 Description of the Event Date of event 2023-02-28 What happened, where and when?',
 'Tropical Cyclone Freddy was one of the longest-lived systems in the Southern Hemisphere.',
 'Freddy formed off the coast of Indonesia in early February 2023 and crossed the southern Indian Ocean, reaching Mauritius and La Réunion.',
 'During its trajectory, Tropical Cyclone Freddy reached the equivalent of a category 5 cyclone and was the first cyclone to exceed this intensity in 2023.',
 'After bringing heavy rains and winds to the islands of Mauritius and La Réunion, Tropical Cyclone Freddy made landfall on the east coast of Madagascar on 21 February 2023 at around 19:00 (local time).',
 'Tropical Cyclone Freddy weakened from a Category 4 cyclone to a Category 3 cyclone before making landfall, but hit Madagascar with sustained winds of 150km/h.',
 'It made landfall in the north of Mananjary, an area previously hit by two tropical cyclones in February 2022 (Batsirai and Eminati) and by t

In [69]:
labelled_reports_dict["MDRMG020"]=[
    {"reportDate": "2024-04-04",
     "hazards": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Mauritius",
     "region" : None,
     "city" : None,
     "locationAnnotation" : ["After bringing heavy rains and winds to the islands of Mauritius and La Réunion, Tropical Cyclone Freddy made landfall on the east coast of Madagascar on 21 February 2023 at around 19:00 (local time).",
                             ],
     "startYear" : 2023,
     "startMonth" : 2,
     "startDay" : 21,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Tropical Cyclone Freddy",
    },
    {"reportDate": "2024-04-04",
     "hazards": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "La Réunion",
     "region" : None,
     "city" : None,
     "locationAnnotation" :["After bringing heavy rains and winds to the islands of Mauritius and La Réunion, Tropical Cyclone Freddy made landfall on the east coast of Madagascar on 21 February 2023 at around 19:00 (local time).",
                             ],
     "startYear" : 2023,
     "startMonth" : 2,
     "startDay" : 21,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Tropical Cyclone Freddy",
    },
    {"reportDate": "2024-04-04",
     "hazards": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Madagascar",
     "region" : ["Mananjary"],
     "city" : None,
     "locationAnnotation" : ["After bringing heavy rains and winds to the islands of Mauritius and La Réunion, Tropical Cyclone Freddy made landfall on the east coast of Madagascar on 21 February 2023 at around 19:00 (local time).",
                             "It made landfall in the north of Mananjary, an area previously hit by two tropical cyclones in February 2022 (Batsirai and Eminati) and by the previous cyclone, Cheneso, a few weeks earlier (January 2023)."],
     "startYear" : 2023,
     "startMonth" : 2,
     "startDay" : 21,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Tropical Cyclone Freddy",
    },
    {"reportDate": "2024-04-04",
     "hazards": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Madagascar",
     "region" : ["Mananjary"],
     "city" : None,
     "locationAnnotation" : ["It made landfall in the north of Mananjary, an area previously hit by two tropical cyclones in February 2022 (Batsirai and Eminati) and by the previous cyclone, Cheneso, a few weeks earlier (January 2023)."],
     "startYear" : 2022,
     "startMonth" : 2,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Tropical Cyclone Batsirai",
    },
    {"reportDate": "2024-04-04",
     "hazards": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Madagascar",
     "region" : ["Mananjary"],
     "city" : None,
     "locationAnnotation" : ["It made landfall in the north of Mananjary, an area previously hit by two tropical cyclones in February 2022 (Batsirai and Eminati) and by the previous cyclone, Cheneso, a few weeks earlier (January 2023)."],
     "startYear" : 2022,
     "startMonth" : 2,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Tropical Cyclone Eminati",
    },
    {"reportDate": "2024-04-04",
     "hazards": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Madagascar",
     "region" : ["Mananjary"],
     "city" : None,
     "locationAnnotation" : ["It made landfall in the north of Mananjary, an area previously hit by two tropical cyclones in February 2022 (Batsirai and Eminati) and by the previous cyclone, Cheneso, a few weeks earlier (January 2023)."],
     "startYear" : 2023,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Tropical Cyclone Cheneso",
    },


]

In [70]:
i=8
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRMY003: 2017-11-21 00:00:00


['Situation analysis Description of the disaster Heavy rains that started in December 2016 continued until late January 2017 in parts of Malaysia, causing flooding in seven states of Peninsular Malaysia – Johor, Kelantan, Pahang, Perak, Terengganu, Malacca and Selangor – and Sabah in East Malaysia.',
 'More than 23,000 people, mainly from smaller towns and villages in rural areas, had to leave their homes to established relief centres.',
 'The situation improved significantly after the weekend of Lunar New Year (28-29 January), with floodwater receding in several affected districts, allowing families that were in relief centres to return home.',
 'National Agency for Disaster Administration (NADMA) reported that the state of Johor suffered the brunt of rising waters, with more than 8,000 evacuees and a fatality.',
 'More information on the floods can be obtained from Information Bulletin n°1 (issued on 5 January), Information Bulletin n°2 (issued on 27 January) and Information Bulletin

In [71]:
labelled_reports_dict["MDRMY003"]=[
    {"reportDate": "2017-11-21",
     "hazards": "Flood",
     "hazardSubtypes" : None,
     "country" : "Malaysia",
     "region" : ["Johor",
                 "Kelantan",
                 "Pahang",
                 "Perak",
                 "Terengganu",
                 "Malacca",
                 "Selangor",
                 "Sabah"],
     "city" : None,
     "locationAnnotation" : ["Situation analysis Description of the disaster Heavy rains that started in December 2016 continued until late January 2017 in parts of Malaysia, causing flooding in seven states of Peninsular Malaysia – Johor, Kelantan, Pahang, Perak, Terengganu, Malacca and Selangor – and Sabah in East Malaysia.",
                             ],
     "startYear" : 2016,
     "startMonth" : 12,
     "startDay" : None,
     "endYear" : 2017,
     "endMonth" : 1,
     "endDay" : 29,
     "hazardName" : None,
    },

]

In [72]:
i=9
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRNI012: 2023-04-06 00:00:00


['The remaining balance of CHF 30,083 will be returned to the Disaster Response Emergency Fund.',
 'The major donors and partners of the Disaster Response Emergency Fund (DREF) include the Red Cross Societies and governments of Belgium, Britain, Canada, Denmark, German, Ireland, Italy, Japan, Luxembourg, New Zealand, Norway, Republic of Korea, Spain, Sweden and Switzerland, as well as DG ECHO, Blizzard Entertainment, Mondelez International Foundation, Fortive Corporation and other corporate and private donors.',
 'The IFRC, on behalf of the Nicaraguan Red Cross, would like to extend thanks to all for their generous contributions.',
 '<Click here for the final financial report and here for the contact information.> A.',
 'SITUATION ANALYSIS Description of the disaster Tropical cyclones are among the natural events that cause the most damage to the population.',
 'Their impact on communities depends on the level of risk to which they are exposed and the level of vulnerability.',
 'Histor

In [73]:
labelled_reports_dict["MDRNI012"]=[
    {"reportDate": "2023-04-06",
     "hazards": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Nicaragua",
     "region" : None,
     "city" : None,
     "locationAnnotation" : ["Historically, Nicaragua has been impacted by tropical cyclones, the most recent being tropical storm Bonnie in May 2022 and category 1 hurricane Julia in October 2022.",
                             ],
     "startYear" : 2022,
     "startMonth" : 5,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "tropical storm Bonnie",
    },
    {"reportDate": "2023-04-06",
     "hazards": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Nicaragua",
     "region" : ["Rivas",
                 "Diriamba",
                 "Managua",
                 "San Rafael del Norte",
                 "Chontales",
                  "Boaco",
                  "Jinotega",
                  "Rivas",
                  "Matagalpa",
                  "León"],
     "city" : None,
     "locationAnnotation" : ["Historically, Nicaragua has been impacted by tropical cyclones, the most recent being tropical storm Bonnie in May 2022 and category 1 hurricane Julia in October 2022.",
                             "The devastating forces of Julia damaged several schools in Rivas, Diriamba, Managua and San Rafael del Norte.",
                             "The areas of Chontales, Boaco, Jinotega, Rivas, Matagalpa and León were among the worst affected regions."],
     "startYear" : 2022,
     "startMonth" : 10,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "hurricane Julia",
    },
    {"reportDate": "2023-04-06",
     "hazards": "Flood",
     "hazardSubtypes" : ["riverine flood"],
     "country" : "Nicaragua",
     "region" : ["Artiwas river",
                 "Wasminona river",
                 "Okonwas river",
                 "Malacatoya river",
                 "Fonseca river",
                 "Siquia river",
                 "Mico river",
                 "Rama river",
                 "South Atlantic Autonomous Region (RAAS)",
                 "Central Zelaya",
                 "Boaco",
                 "South Caribbean"],
     "city" : ["Rosita",
               "El Rama"],
     "locationAnnotation" : ["Julia caused heavy rainfall, which in turn caused several rivers to overflow, including the Artiwas, Wasminona and Okonwas rivers in the municipality of Rosita, the Malacatoya and Fonseca rivers, Siquia, Mico and Rama, among others, putting the population at risk and damaging social infrastructure such as housing, roads and telecommunications in the South Atlantic Autonomous Region (RAAS), Central Zelaya and Boaco, interruption of electricity and drinking water services, obstruction of roads due to falling trees, among others.",
                             "One of the areas most affected was the municipality of El Rama in the South Caribbean, where three rivers converge: Siquia, Mico and Rama, adding to the more than 70 rivers that overflowed nationwide as a result of the rains, leaving villages under water and entire families lost all their belongings."],
     "startYear" : 2022,
     "startMonth" : 10,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "hurricane Julia",
    },
    {"reportDate": "2023-04-06",
     "hazards": "Mass Movement",
     "hazardSubtypes" : ["landslide"],
     "country" : "Nicaragua",
     "region" : None,
     "city" : None,
     "locationAnnotation" : None,
     "startYear" : 2022,
     "startMonth" : 10,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "hurricane Julia",
    },

]

In [74]:
i=10
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRS2001: 2024-08-20 00:00:00


['SITUATION ANALYSIS Description of the crisis Hurricane Beryl emerged as a significant climate event, developing from a monitored tropical wave on June 25, 2024.',
 'The storm rapidly intensified, becoming the first major hurricane of the 2024 Atlantic season and reaching unprecedented strength.',
 'By June 29, 2024, Beryl had attained Category 4 status, setting a record as the earliest Category 4 hurricane in history.',
 'The storm continued to strengthen, reaching Category 5 with maximum sustained winds of 270 km/h by July 1, 2024.',
 'This highlights the increasing severity and unpredictability of hurricanes in the Caribbean, exacerbated by rising sea temperatures.',
 'On July 1, 2024, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.',
 'In Grenada, more than 1,600 people sought refuge in emergency shelters, and over 98% of buildings on Carriacou and Petit Martinique suffered severe damage.',
 'The destruction extended to critical i

In [75]:
labelled_reports_dict["MDRS2001"]=[
    {"reportDate": "2024-08-20",
     "hazards": "Storm",
     "hazardSubtypes" : ["tropical storm", "storm surge"],
     "country" : "Grenada",
     "region" : ["Carriacou",
                 "Petit Martinique"],
     "city" : None,
     "locationAnnotation" : ["On July 1, 2024, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.",
                             "In Grenada, more than 1,600 people sought refuge in emergency shelters, and over 98% of buildings on Carriacou and Petit Martinique suffered severe damage.",
                             ],
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Hurricane Beryl",
    },
    {"reportDate": "2024-08-20",
     "hazards": "Storm",
     "hazardSubtypes" : ["tropical storm", "storm surge"],
     "country" : "Saint Vincent and the Grenadines",
     "region" : ["Union Island"],
     "city" : None,
     "locationAnnotation" : ["On July 1, 2024, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.",
                             "Saint Vincent and the Grenadines experienced similar devastation, with 90% of homes on Union Island damaged or destroyed, impacting essential services and leaving many residents without access to healthcare and other services."],
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Hurricane Beryl",
    },
    {"reportDate": "2024-08-20",
     "hazards": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Jamaica",
     "region" : ["Clarendon",
                 "St. Elizabeth"],
     "city" : ["St. Thomas",
               "Manchester",
               "Westmoreland",
               "Hanover"],
     "locationAnnotation" : ["1 Situation Report 11 - Hurricane Beryl - Grenada and St. Vincent and the Grenadines - 29 July 2024 - PAHO/WHO | Pan American Health Organization Operations Update-2 3 Jamaica experienced widespread damage as Hurricane Beryl once again intensified to Category 5, severely impacting infrastructure and agriculture.",
                             "The hardest-hit areas included Clarendon and St. Elizabeth, with extensive damage reported in St. Thomas, Manchester, Westmoreland, and Hanover.",
                             ],
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Hurricane Beryl",
    },


]

In [76]:
i=11
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRSV012: 2019-06-26 00:00:00


['The major donors and partners of the Disaster Relief Emergency Fund (DREF) include the Red Cross Societies and governments of Belgium, Britain, Canada, Denmark, German, Ireland, Italy, Japan, Luxembourg, New Zealand, Norway, Republic of Korea, Spain, Sweden and Switzerland, as well as DG ECHO and Blizzard Entertainment, Mondelez International Foundation, and Fortive Corporation and other corporate and private donors.',
 'The IFRC, on behalf of the national society, would like to extend thanks to all for their generous contributions.',
 'ECHO and the government of Canada have replenished the DREF in the occasion of this operation.',
 'The total amount spent under this DREF operation was 118,630 CHF.',
 'The remaining balance of 32,041 CHF will be reimbursed to the Disaster Relief Emergency Fund.',
 '< For the Final Financial Report, click here.',
 'For contact information, click here.',
 'Situation analysis Description of the disaster On 6 October, rains began falling over eastern El 

In [77]:
labelled_reports_dict["MDRSV012"]=[
    {"reportDate": "2019-06-26",
     "hazards": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "El Salvador",
     "region" : ["Yucatán channel",
                 "Morazán department",
                 "La Union department",
                 "Sonsonate department",
                 "eastern region"
                 "western El Salvador"],
     "city" : ["El Brazo canton",
               "La Canoa canton",
               "El Tecomatal canton",
               "San Miguel municipality",
               "San Felipe canton",
               "Las Tunas canton",
               "Capitán Lazo canton",
               "Puerto Parada canton",
               "Usulután municipality",
               "Metalío canton"],
     "locationAnnotation" : ["Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No. 14 located near the Honduran Atlantic coast.",
                             "On 7 October, the tropical depression was upgraded to Tropical Storm Michael, which continued moving north over the Yucatán channel toward the System declared a Green Alert for the entire country.",
                             "On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazán department and two in La Union department.",
                             "The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel; the cantons of San Felipe and Las Tunas in La Unión department; the cantons of Capitán Lazo and Puerto Parada in the municipality of Usulután; as well as the canton of Metalío in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador: Floods Photo: Area affected by Hurricane Michael",
                             ],
     "startYear" : 2018,
     "startMonth" : 10,
     "startDay" : 6,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Tropical Storm Michael",
    },
    {"reportDate": "2019-06-26",
     "hazards": "Flood",
     "hazardSubtypes" : None,
     "country" : "El Salvador",
     "region" : ["Yucatán channel",
                 "Morazán department",
                 "La Union department",
                 "Sonsonate department",
                 "eastern region"
                 "western El Salvador"],
     "city" : ["El Brazo canton",
               "La Canoa canton",
               "El Tecomatal canton",
               "San Miguel municipality",
               "San Felipe canton",
               "Las Tunas canton",
               "Capitán Lazo canton",
               "Puerto Parada canton",
               "Usulután municipality",
               "Metalío canton"],
     "locationAnnotation" : ["Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No. 14 located near the Honduran Atlantic coast.",
                             "On 7 October, the tropical depression was upgraded to Tropical Storm Michael, which continued moving north over the Yucatán channel toward the System declared a Green Alert for the entire country.",
                             "On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazán department and two in La Union department.",
                             "The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel; the cantons of San Felipe and Las Tunas in La Unión department; the cantons of Capitán Lazo and Puerto Parada in the municipality of Usulután; as well as the canton of Metalío in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador: Floods Photo: Area affected by Hurricane Michael",
                             ],
     "startYear" : 2018,
     "startMonth" : 10,
     "startDay" : 6,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Tropical Storm Michael",
    },

]

In [78]:
i=12
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRYE011: 2023-05-29 00:00:00


["SITUATION ANALYSIS Description of the disaster Heavy rains in Sana'a governorate caused extensive damage to public infrastructure, shelters for displaced people, and other private property.",
 'Three people died and two were injured due to the heavy rain.',
 "In Sa'adah Governorate, 299 families were affected, and approximately 50 of them were affected by heavy rains in various districts.",
 "Yemen's annual rainy season starts in May and normally goes until August or September, but in 2022, Yemen witnessed heavier than normal rains, ranging in intensity, accompanied by thunderstorms starting in May 2022.",
 'According to the Food and Agriculture Organization of the United Nations (FAO) Agrometeorological Early Warning Bulletin1, the forecasts covering until July 31 favour the formation of heavy rains, especially affecting areas to the north of Ibb and central Hadramawt.',
 'Heavy rains and flooding caused significant damage in Yemen from May to mid-September 2022, resulting in the lo

In [79]:
labelled_reports_dict["MDRYE011"]=[
    {"reportDate": "2023-05-29",
     "hazards": "Flood",
     "hazardSubtypes" : ["flash flood"],
     "country" : "Yemen",
     "region" : ["Sana'a governorate",
                 "Sa'adah governorate",
                 "north of Ibb",
                 "central Hadramawt",
                 "Sanaa"],
     "city" : ["Gaps Ad Dali'",
               "Al Bayda",
               "Al Hodeidah",
               "Al Jawf",
               "Al Maharah",
               "Al Mahwit",
               "Amran",
               "Dhamar",
               "Hadramawt",
               "Hajjah",
               "Ibb",
               "Ma'rib",
               "Sa'dah",
               "Sana'a City",
               "Shabwah"],
     "locationAnnotation" : ["SITUATION ANALYSIS Description of the disaster Heavy rains in Sana'a governorate caused extensive damage to public infrastructure, shelters for displaced people, and other private property.",
                             "In Sa'adah Governorate, 299 families were affected, and approximately 50 of them were affected by heavy rains in various districts.",
                             "Yemen's annual rainy season starts in May and normally goes until August or September, but in 2022, Yemen witnessed heavier than normal rains, ranging in intensity, accompanied by thunderstorms starting in May 2022.",
                             "According to the Food and Agriculture Organization of the United Nations (FAO) Agrometeorological Early Warning Bulletin1, the forecasts covering until July 31 favour the formation of heavy rains, especially affecting areas to the north of Ibb and central Hadramawt.",
                             "Heavy rains and flooding caused significant damage in Yemen from May to mid-September 2022, resulting in the loss of lives, destruction of property and livelihoods, and damage to infrastructure.",
                             "1 Agrometeorological Early Warning Bulletin (18-31 July 2022) [EN/AR] - Yemen | ReliefWeb Final Report Yemen: Sanaa Floods Internal Internal Internal Heavy rains and flooding continued across Yemen into the third week of August 2022.",
                             "©YRCS Internal Internal Internal Governorate Total Affected HHs since start of 2022 floods Total HHs Reached by YRCS since start of the 2022 flood Gaps Ad Dali' 647 647 0 Al Bayda 839 448 391 Al Hodeidah 1031 634 397 Al Jawf 4291 250 4041 Al Maharah 53 53 0 Al Mahwit 162 79 83 Amran 2688 497 2191 Dhamar 520 69 451 Hadramawt 1112 112 1000 Hajjah 1936 1031 905 Ibb 719 210 509 Ma'rib 23731 1750 21981 Sa'dah 229 170 59 Sana'a City 783 555 228 Sana'a Governorate 1121 407 714 Shabwah 392 1 391 Total 40,254 6,913 33,341 Of the total number of people affected, an estimated 17,000 across affected IDPs sites have suffered total damages to tents and other belongings.",
                            ],
     "startYear" : 2022,
     "startMonth" : 5,
     "startDay" : None,
     "endYear" : 2022,
     "endMonth" : 9,
     "endDay" : 15,
     "hazardName" : None,
    },
    {"reportDate": "2023-05-29",
     "hazards": "Storm",
     "hazardSubtypes" : None,
     "country" : "Yemen",
     "region" : ["Sana'a governorate",
                 "Sa'adah governorate",
                 "north of Ibb",
                 "central Hadramawt",
                 "Sanaa"],
     "city" : ["Gaps Ad Dali'",
               "Al Bayda",
               "Al Hodeidah",
               "Al Jawf",
               "Al Maharah",
               "Al Mahwit",
               "Amran",
               "Dhamar",
               "Hadramawt",
               "Hajjah",
               "Ibb",
               "Ma'rib",
               "Sa'dah",
               "Sana'a City",
               "Shabwah"],
     "locationAnnotation" : ["SITUATION ANALYSIS Description of the disaster Heavy rains in Sana'a governorate caused extensive damage to public infrastructure, shelters for displaced people, and other private property.",
                             "In Sa'adah Governorate, 299 families were affected, and approximately 50 of them were affected by heavy rains in various districts.",
                             "Yemen's annual rainy season starts in May and normally goes until August or September, but in 2022, Yemen witnessed heavier than normal rains, ranging in intensity, accompanied by thunderstorms starting in May 2022.",
                             "According to the Food and Agriculture Organization of the United Nations (FAO) Agrometeorological Early Warning Bulletin1, the forecasts covering until July 31 favour the formation of heavy rains, especially affecting areas to the north of Ibb and central Hadramawt.",
                             "Heavy rains and flooding caused significant damage in Yemen from May to mid-September 2022, resulting in the loss of lives, destruction of property and livelihoods, and damage to infrastructure.",
                             "1 Agrometeorological Early Warning Bulletin (18-31 July 2022) [EN/AR] - Yemen | ReliefWeb Final Report Yemen: Sanaa Floods Internal Internal Internal Heavy rains and flooding continued across Yemen into the third week of August 2022.",
                             "©YRCS Internal Internal Internal Governorate Total Affected HHs since start of 2022 floods Total HHs Reached by YRCS since start of the 2022 flood Gaps Ad Dali' 647 647 0 Al Bayda 839 448 391 Al Hodeidah 1031 634 397 Al Jawf 4291 250 4041 Al Maharah 53 53 0 Al Mahwit 162 79 83 Amran 2688 497 2191 Dhamar 520 69 451 Hadramawt 1112 112 1000 Hajjah 1936 1031 905 Ibb 719 210 509 Ma'rib 23731 1750 21981 Sa'dah 229 170 59 Sana'a City 783 555 228 Sana'a Governorate 1121 407 714 Shabwah 392 1 391 Total 40,254 6,913 33,341 Of the total number of people affected, an estimated 17,000 across affected IDPs sites have suffered total damages to tents and other belongings.",
                            ],
     "startYear" : 2022,
     "startMonth" : 5,
     "startDay" : None,
     "endYear" : 2022,
     "endMonth" : 9,
     "endDay" : 15,
     "hazardName" : None,
    },
]

In [ ]:
labelled_impact_reports_dict["MDRGN015"]=[
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Affected People",
     "impactValue" : 24135, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Page 1 19 DREF Operational Update Guinea Floods Appeal MDRGN015 Total DREF Allocation Crisis Category Yellow Hazard Flood Glide Number FL2023000158GIN People Affected 24,135 people People Targeted 14,350 people Event Onset Sudden Operation Start Date 20230823 New Operational End Date 20240131 Total Operating Timeframe 5 months Additional Allocation Requested Targeted Areas Kindia Page 2 19 Description of the Event What happened, where and when?'],
     "country" : ["Guinea"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 23,
     "endYear" : 2024,
     "endMonth" : 1,
     "endDay" : 31,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Road Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Major roads in Conakry and Siguiri were rendered impassable due to the flood waters, heavily constraining vehicles, and pedestrians having to find alternative routes.'],
     "country" : ["Guinea"],
     "location" : ["Conakry", "Siguiri"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 23,
     "endYear" : 2024,
     "endMonth" : 1,
     "endDay" : 31,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Mobility and Access to Transport",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Major roads in Conakry and Siguiri were rendered impassable due to the flood waters, heavily constraining vehicles, and pedestrians having to find alternative routes.'],
     "country" : ["Guinea"],
     "location" : ["Conakry", "Siguiri"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 23,
     "endYear" : 2024,
     "endMonth" : 1,
     "endDay" : 31,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Economical disruption",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Furthermore, the areas affected by the floods have already been experiencing acute humanitarian needs due to economic hardship and financial instability.'],
     "country" : ["Guinea"],
     "location" : ["Conakry", "Siguiri", "Coyah"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Other Infrastructure impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['These rains can cause further damage to infrastructure, road closures, and contamination.'],
     "country" : ["Guinea"],
     "location" : ["Conakry", "Siguiri", "Coyah"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Road Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['These rains can cause further damage to infrastructure, road closures, and contamination.'],
     "country" : ["Guinea"],
     "location" : ["Conakry", "Siguiri", "Coyah"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Ill People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['These rains can cause further damage to infrastructure, road closures, and contamination.'],
     "country" : ["Guinea"],
     "location" : ["Conakry", "Siguiri", "Coyah"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Page 3 19 Field mission with IFRC, Guinea Red Cross, and Agence Nationale de Gestion des Urgences et Catastrophes Humanitaires National Agency for Emergency and Humanitarian Disaster Management Scope and Scale August has recorded significant heavy rains across Guinea, affecting several prefectures of the country On August 6, 2023 heavy rains in the prefecture of Coyah and its surroundings overflowed the Sarinka River, resulting in flooding in the Urban and Rural Communities of Maneah and Kouriah, and affecting nine localities namely Fily, Tougand, Batouyah, and Sambaya in the urban communities Tann and Tann 2 in the Rural Commune of Kouria and Boguitagui, Bangouyah, and Kouria center in the Rural Commune of Kouria, destroying several homes and washing away properties of the affected families.'],
     "country" : ["Guinea"],
     "location" : ["Coyah", "Sarinka River", "Maneah", "Kouriah", "Fily", "Tougand", "Batouyah", "Sambaya", "Tann", "Tann 2", "Kouria", "Boguitagui", "Bangouyah", "Kouria"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 6,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Page 3 19 Field mission with IFRC, Guinea Red Cross, and Agence Nationale de Gestion des Urgences et Catastrophes Humanitaires National Agency for Emergency and Humanitarian Disaster Management Scope and Scale August has recorded significant heavy rains across Guinea, affecting several prefectures of the country On August 6, 2023 heavy rains in the prefecture of Coyah and its surroundings overflowed the Sarinka River, resulting in flooding in the Urban and Rural Communities of Maneah and Kouriah, and affecting nine localities namely Fily, Tougand, Batouyah, and Sambaya in the urban communities Tann and Tann 2 in the Rural Commune of Kouria and Boguitagui, Bangouyah, and Kouria center in the Rural Commune of Kouria, destroying several homes and washing away properties of the affected families.'],
     "country" : ["Guinea"],
     "location" : ["Coyah", "Sarinka River", "Maneah", "Kouriah", "Fily", "Tougand", "Batouyah", "Sambaya", "Tann", "Tann 2", "Kouria", "Boguitagui", "Bangouyah", "Kouria"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 6,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Injured People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Page 3 19 Field mission with IFRC, Guinea Red Cross, and Agence Nationale de Gestion des Urgences et Catastrophes Humanitaires National Agency for Emergency and Humanitarian Disaster Management Scope and Scale August has recorded significant heavy rains across Guinea, affecting several prefectures of the country On August 6, 2023 heavy rains in the prefecture of Coyah and its surroundings overflowed the Sarinka River, resulting in flooding in the Urban and Rural Communities of Maneah and Kouriah, and affecting nine localities namely Fily, Tougand, Batouyah, and Sambaya in the urban communities Tann and Tann 2 in the Rural Commune of Kouria and Boguitagui, Bangouyah, and Kouria center in the Rural Commune of Kouria, destroying several homes and washing away properties of the affected families.', 
                     'During evacuation and rescue, some women, children, and the elderly were identified as requiring urgent medical support, while others were trapped in their houses, and some sat on the roof of their houses requesting rescue.'],
     "country" : ["Guinea"],
     "location" : ["Coyah", "Sarinka River", "Maneah", "Kouriah", "Fily", "Tougand", "Batouyah", "Sambaya", "Tann", "Tann 2", "Kouria", "Boguitagui", "Bangouyah", "Kouria"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 6,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Mobility and Access to Transport",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Page 3 19 Field mission with IFRC, Guinea Red Cross, and Agence Nationale de Gestion des Urgences et Catastrophes Humanitaires National Agency for Emergency and Humanitarian Disaster Management Scope and Scale August has recorded significant heavy rains across Guinea, affecting several prefectures of the country On August 6, 2023 heavy rains in the prefecture of Coyah and its surroundings overflowed the Sarinka River, resulting in flooding in the Urban and Rural Communities of Maneah and Kouriah, and affecting nine localities namely Fily, Tougand, Batouyah, and Sambaya in the urban communities Tann and Tann 2 in the Rural Commune of Kouria and Boguitagui, Bangouyah, and Kouria center in the Rural Commune of Kouria, destroying several homes and washing away properties of the affected families.', 
                     'During evacuation and rescue, some women, children, and the elderly were identified as requiring urgent medical support, while others were trapped in their houses, and some sat on the roof of their houses requesting rescue.'],
     "country" : ["Guinea"],
     "location" : ["Coyah", "Sarinka River", "Maneah", "Kouriah", "Fily", "Tougand", "Batouyah", "Sambaya", "Tann", "Tann 2", "Kouria", "Boguitagui", "Bangouyah", "Kouria"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 6,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 3, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The damages registered are as follows In Coyah the initial assessments provided by the Guinea Red Cross indicate floods reportedly killed three people, leaving 933 people injured.'],
     "country" : ["Guinea"],
     "location" : ["Coyah"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Injured People",
     "impactValue" : 933, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The damages registered are as follows In Coyah the initial assessments provided by the Guinea Red Cross indicate floods reportedly killed three people, leaving 933 people injured.'],
     "country" : ["Guinea"],
     "location" : ["Coyah"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Displaced People",
     "impactValue" : 24135, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Some 3,448 households that is 24,135 people have been reported to be displaced.'],
     "country" : ["Guinea"],
     "location" : ["Coyah"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 1, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['In Siguiri the floods caused 1 death, 4 seriously injured, 46 people rescued.', 
                     'In Siguiri, six localities have been affected neighborhoods of Siguiri Koura 2, Bolibana, Hrmakono, Kouroudakoro, the district of Balato Subprefecture of Kintinian and Malah center Subprefecture of Maleah.'],
     "country" : ["Guinea"],
     "location" : ["Siguiri", "Siguiri Koura 2", "Bolibana", "Hrmakono", "Kouroudakoro", "Balato", "Kintinian", "Malah"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Injured People",
     "impactValue" : 4, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['In Siguiri the floods caused 1 death, 4 seriously injured, 46 people rescued.', 
                     'In Siguiri, six localities have been affected neighborhoods of Siguiri Koura 2, Bolibana, Hrmakono, Kouroudakoro, the district of Balato Subprefecture of Kintinian and Malah center Subprefecture of Maleah.'],
     "country" : ["Guinea"],
     "location" : ["Siguiri", "Siguiri Koura 2", "Bolibana", "Hrmakono", "Kouroudakoro", "Balato", "Kintinian", "Malah"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Displaced People",
     "impactValue" : 46, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['In Siguiri the floods caused 1 death, 4 seriously injured, 46 people rescued.', 
                     'In Siguiri, six localities have been affected neighborhoods of Siguiri Koura 2, Bolibana, Hrmakono, Kouroudakoro, the district of Balato Subprefecture of Kintinian and Malah center Subprefecture of Maleah.'],
     "country" : ["Guinea"],
     "location" : ["Siguiri", "Siguiri Koura 2", "Bolibana", "Hrmakono", "Kouroudakoro", "Balato", "Kintinian", "Malah"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Affected People",
     "impactValue" : 18326, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['About 2,618 families that is 18,326 people have been affected including 7,330 displaced people, 300 latrines and 241 houses destroyed, 52 shops and 3 service stations impacted.', 
                     'In Siguiri, six localities have been affected neighborhoods of Siguiri Koura 2, Bolibana, Hrmakono, Kouroudakoro, the district of Balato Subprefecture of Kintinian and Malah center Subprefecture of Maleah.'],
     "country" : ["Guinea"],
     "location" : ["Siguiri", "Siguiri Koura 2", "Bolibana", "Hrmakono", "Kouroudakoro", "Balato", "Kintinian", "Malah"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Displaced People",
     "impactValue" : 7330, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['About 2,618 families that is 18,326 people have been affected including 7,330 displaced people, 300 latrines and 241 houses destroyed, 52 shops and 3 service stations impacted.', 
                     'In Siguiri, six localities have been affected neighborhoods of Siguiri Koura 2, Bolibana, Hrmakono, Kouroudakoro, the district of Balato Subprefecture of Kintinian and Malah center Subprefecture of Maleah.'],
     "country" : ["Guinea"],
     "location" : ["Siguiri", "Siguiri Koura 2", "Bolibana", "Hrmakono", "Kouroudakoro", "Balato", "Kintinian", "Malah"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Water, Sanitation, and Hygiene Infrastructure",
     "impactValue" : 300, 
     "impactUnit" : "latrines", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['About 2,618 families that is 18,326 people have been affected including 7,330 displaced people, 300 latrines and 241 houses destroyed, 52 shops and 3 service stations impacted.', 
                     'In Siguiri, six localities have been affected neighborhoods of Siguiri Koura 2, Bolibana, Hrmakono, Kouroudakoro, the district of Balato Subprefecture of Kintinian and Malah center Subprefecture of Maleah.'],
     "country" : ["Guinea"],
     "location" : ["Siguiri", "Siguiri Koura 2", "Bolibana", "Hrmakono", "Kouroudakoro", "Balato", "Kintinian", "Malah"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 241, 
     "impactUnit" : "houses", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['About 2,618 families that is 18,326 people have been affected including 7,330 displaced people, 300 latrines and 241 houses destroyed, 52 shops and 3 service stations impacted.', 
                     'In Siguiri, six localities have been affected neighborhoods of Siguiri Koura 2, Bolibana, Hrmakono, Kouroudakoro, the district of Balato Subprefecture of Kintinian and Malah center Subprefecture of Maleah.'],
     "country" : ["Guinea"],
     "location" : ["Siguiri", "Siguiri Koura 2", "Bolibana", "Hrmakono", "Kouroudakoro", "Balato", "Kintinian", "Malah"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Other Infrastructure impact",
     "impactValue" : 52, 
     "impactUnit" : "shops", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['About 2,618 families that is 18,326 people have been affected including 7,330 displaced people, 300 latrines and 241 houses destroyed, 52 shops and 3 service stations impacted.', 
                     'In Siguiri, six localities have been affected neighborhoods of Siguiri Koura 2, Bolibana, Hrmakono, Kouroudakoro, the district of Balato Subprefecture of Kintinian and Malah center Subprefecture of Maleah.'],
     "country" : ["Guinea"],
     "location" : ["Siguiri", "Siguiri Koura 2", "Bolibana", "Hrmakono", "Kouroudakoro", "Balato", "Kintinian", "Malah"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Other Transportation Infrastructure",
     "impactValue" : 3, 
     "impactUnit" : "service stations", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['About 2,618 families that is 18,326 people have been affected including 7,330 displaced people, 300 latrines and 241 houses destroyed, 52 shops and 3 service stations impacted.', 
                     'In Siguiri, six localities have been affected neighborhoods of Siguiri Koura 2, Bolibana, Hrmakono, Kouroudakoro, the district of Balato Subprefecture of Kintinian and Malah center Subprefecture of Maleah.'],
     "country" : ["Guinea"],
     "location" : ["Siguiri", "Siguiri Koura 2", "Bolibana", "Hrmakono", "Kouroudakoro", "Balato", "Kintinian", "Malah"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 3, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['In Conakry 4 deaths have been reported.', 
                     'In Conakry, eight 8 localities in the communes of Ratoma and Matoto in Conakry Wareah Kinifi Soloprimo, Kinifi village, Nriboundji, Kobaya bafond Yattaya Fossidet bridge and Lambanyi Africof, Enta market, lowland cemetery sector, Lansanaya dam, and Kissoso.',],
     "country" : ["Guinea"],
     "location" : ["Conakry", "Ratoma", "Matoto", "Conakry Wareah Kinifi Soloprimo", "Kinifi village", "Nriboundji", "Kobaya bafond Yattaya Fossidet bridge", "Lambanyi Africof", "Enta market", "lowland cemetery sector", "Lansanaya dam", "Kissoso"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Affected People",
     "impactValue" : 8491, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Some 1213 households that is 8,491 people are affected including 3,396 displaced Page 4 19 people, 172 houses and 230 latrines destroyed, and 28 water points been impacted.', 
                     'In Conakry, eight 8 localities in the communes of Ratoma and Matoto in Conakry Wareah Kinifi Soloprimo, Kinifi village, Nriboundji, Kobaya bafond Yattaya Fossidet bridge and Lambanyi Africof, Enta market, lowland cemetery sector, Lansanaya dam, and Kissoso.',],
     "country" : ["Guinea"],
     "location" : ["Conakry", "Ratoma", "Matoto", "Conakry Wareah Kinifi Soloprimo", "Kinifi village", "Nriboundji", "Kobaya bafond Yattaya Fossidet bridge", "Lambanyi Africof", "Enta market", "lowland cemetery sector", "Lansanaya dam", "Kissoso"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Displaced People",
     "impactValue" : 3396, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Some 1213 households that is 8,491 people are affected including 3,396 displaced Page 4 19 people, 172 houses and 230 latrines destroyed, and 28 water points been impacted.', 
                     'In Conakry, eight 8 localities in the communes of Ratoma and Matoto in Conakry Wareah Kinifi Soloprimo, Kinifi village, Nriboundji, Kobaya bafond Yattaya Fossidet bridge and Lambanyi Africof, Enta market, lowland cemetery sector, Lansanaya dam, and Kissoso.',],
     "country" : ["Guinea"],
     "location" : ["Conakry", "Ratoma", "Matoto", "Conakry Wareah Kinifi Soloprimo", "Kinifi village", "Nriboundji", "Kobaya bafond Yattaya Fossidet bridge", "Lambanyi Africof", "Enta market", "lowland cemetery sector", "Lansanaya dam", "Kissoso"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 172, 
     "impactUnit" : "houses", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Some 1213 households that is 8,491 people are affected including 3,396 displaced Page 4 19 people, 172 houses and 230 latrines destroyed, and 28 water points been impacted.', 
                     'In Conakry, eight 8 localities in the communes of Ratoma and Matoto in Conakry Wareah Kinifi Soloprimo, Kinifi village, Nriboundji, Kobaya bafond Yattaya Fossidet bridge and Lambanyi Africof, Enta market, lowland cemetery sector, Lansanaya dam, and Kissoso.',],
     "country" : ["Guinea"],
     "location" : ["Conakry", "Ratoma", "Matoto", "Conakry Wareah Kinifi Soloprimo", "Kinifi village", "Nriboundji", "Kobaya bafond Yattaya Fossidet bridge", "Lambanyi Africof", "Enta market", "lowland cemetery sector", "Lansanaya dam", "Kissoso"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Water, Sanitation, and Hygiene Infrastructure",
     "impactValue" : 230, 
     "impactUnit" : "latrines", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Some 1213 households that is 8,491 people are affected including 3,396 displaced Page 4 19 people, 172 houses and 230 latrines destroyed, and 28 water points been impacted.', 
                     'In Conakry, eight 8 localities in the communes of Ratoma and Matoto in Conakry Wareah Kinifi Soloprimo, Kinifi village, Nriboundji, Kobaya bafond Yattaya Fossidet bridge and Lambanyi Africof, Enta market, lowland cemetery sector, Lansanaya dam, and Kissoso.',],
     "country" : ["Guinea"],
     "location" : ["Conakry", "Ratoma", "Matoto", "Conakry Wareah Kinifi Soloprimo", "Kinifi village", "Nriboundji", "Kobaya bafond Yattaya Fossidet bridge", "Lambanyi Africof", "Enta market", "lowland cemetery sector", "Lansanaya dam", "Kissoso"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2023-11-30",
     "impactSubtype" : "Water, Sanitation, and Hygiene Infrastructure",
     "impactValue" : 28, 
     "impactUnit" : "water points", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Some 1213 households that is 8,491 people are affected including 3,396 displaced Page 4 19 people, 172 houses and 230 latrines destroyed, and 28 water points been impacted.', 
                     'In Conakry, eight 8 localities in the communes of Ratoma and Matoto in Conakry Wareah Kinifi Soloprimo, Kinifi village, Nriboundji, Kobaya bafond Yattaya Fossidet bridge and Lambanyi Africof, Enta market, lowland cemetery sector, Lansanaya dam, and Kissoso.',],
     "country" : ["Guinea"],
     "location" : ["Conakry", "Ratoma", "Matoto", "Conakry Wareah Kinifi Soloprimo", "Kinifi village", "Nriboundji", "Kobaya bafond Yattaya Fossidet bridge", "Lambanyi Africof", "Enta market", "lowland cemetery sector", "Lansanaya dam", "Kissoso"],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 11,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }
]

In [ ]:
labelled_impact_reports_dict["MDRKE058"]=[
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected People",
     "impactValue" : 281880, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Page 1 21 DREF Operation Kenya Floods Evacuation of the a ected families in Garissa County Appeal MDRKE058Country KenyaHazard FloodType of DREF Response Crisis Category OrangeEvent Onset SlowDREF Allocation CHF 749,939 Glide Number People A ected 281,880 peoplePeople Targeted 150,000 people Operation Start Date 20231111Operation Timeframe 4 monthsOperation End Date 20240331DREF Published 20231115 Targeted Areas Tana River, Garissa, Wajir, Mandera, Marsabit, Isiolo, MeruPage 2 21 Description of the Event KRCS oods damage What happened, where and when?',],
     "country" : ["Kenya"],
     "location" : ["Tana River", "Garissa", "Wajir","Mandera", "Marsabit", "Isiolo", "Meru"],
     "startYear" : 2023,
     "startMonth" : 11,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 3,
     "endDay" : 31,
     "hazards" : ["Flood"]
    }, 
    
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected People",
     "impactValue" : 25030, 
     "impactUnit" : "households", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['3 Description of the crisis The country has been experiencing flood affects since the onset of the March-April-May (MAM) long rains season where 24 out of 47 counties have been affected, 25,030 households were affected and 11,275 households displaced.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Displaced People",
     "impactValue" : 11275, 
     "impactUnit" : "households", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['3 Description of the crisis The country has been experiencing flood affects since the onset of the March-April-May (MAM) long rains season where 24 out of 47 counties have been affected, 25,030 households were affected and 11,275 households displaced.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Other Economic & Livelihood impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The economic activities of the areas have been significantly disrupted with major roads destroyed totalling to 11 critical facilities such as 7 health facilities flooded, 18 schools destroyed, 80 businesses hampered.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Road Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The economic activities of the areas have been significantly disrupted with major roads destroyed totalling to 11 critical facilities such as 7 health facilities flooded, 18 schools destroyed, 80 businesses hampered.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Healthcare Infrastructure",
     "impactValue" : 7, 
     "impactUnit" : "health facilities", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The economic activities of the areas have been significantly disrupted with major roads destroyed totalling to 11 critical facilities such as 7 health facilities flooded, 18 schools destroyed, 80 businesses hampered.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Education Infrastructure",
     "impactValue" : 18, 
     "impactUnit" : "schools", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The economic activities of the areas have been significantly disrupted with major roads destroyed totalling to 11 critical facilities such as 7 health facilities flooded, 18 schools destroyed, 80 businesses hampered.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Other Economic & Livelihood impact",
     "impactValue" : 80, 
     "impactUnit" : "businesses", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The economic activities of the areas have been significantly disrupted with major roads destroyed totalling to 11 critical facilities such as 7 health facilities flooded, 18 schools destroyed, 80 businesses hampered.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected Livestock and Animals",
     "impactValue" : None, 
     "impactUnit" : "livestock", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 3401, 
     "impactValueMax" : None, 
     "annotation" : ['More than 3,401 livestock have been lost and approximately 26,748 acres of crops destroyed.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : 26748, 
     "impactUnit" : "acres of crops", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['More than 3,401 livestock have been lost and approximately 26,748 acres of crops destroyed.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected People",
     "impactValue" : 31015, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['In Nairobi County, 31,015 people (6,203 households) in the informal settlements of Kware, Kibra, Viwandani, Mukuru Kwa Njenga, Kayole, and Mukuru Kwa Reuben were affected by flooding due to poor and blocked drainage systems.'],
     "country" : ["Kenya"],
     "location" : ["Nairobi County", "Kware", "Kibra", "Viwandani", "Mukuru Kwa Njenga", "Kayole", "Mukuru Kwa Reuben"],
     "startYear" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Human Deaths",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 178, 
     "impactValueMax" : None, 
     "annotation" : ['5 The floods are exacerbating the humanitarian crisis in the region just as it emerges from the El Nino which occurred late 2023 when at least 178 people were killed, injured 242 and thousands displaced.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Injured People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 242, 
     "impactValueMax" : None, 
     "annotation" : ['5 The floods are exacerbating the humanitarian crisis in the region just as it emerges from the El Nino which occurred late 2023 when at least 178 people were killed, injured 242 and thousands displaced.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 1000, 
     "impactValueMax" : None, 
     "annotation" : ['5 The floods are exacerbating the humanitarian crisis in the region just as it emerges from the El Nino which occurred late 2023 when at least 178 people were killed, injured 242 and thousands displaced.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "impactUnit" : "households", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 139051, 
     "impactValueMax" : None, 
     "annotation" : ['Over 139,051 households were affected, 64,516 households displaced.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : "households", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 64516, 
     "impactValueMax" : None, 
     "annotation" : ['Over 139,051 households were affected, 64,516 households displaced.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Other Infrastructure impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Destroyed infrastructure, health and educational services and facilities.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Healthcare Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Destroyed infrastructure, health and educational services and facilities.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Education Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Destroyed infrastructure, health and educational services and facilities.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected Livestock and Animals",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Communities reported significant loss of livestock, crops and small businesses leading to loss of livelihood.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Communities reported significant loss of livestock, crops and small businesses leading to loss of livelihood.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Other Economic & Livelihood impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Communities reported significant loss of livestock, crops and small businesses leading to loss of livelihood.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Injured People",
     "impactValue" : 242, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : 10,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Injured People",
     "impactValue" : 27, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 178, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : 10,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 38, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Displaced People",
     "impactValue" : 64519, 
     "impactUnit" : "households", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : 10,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Displaced People",
     "impactValue" : 11275, 
     "impactUnit" : "households", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected People",
     "impactValue" : 139071, 
     "impactUnit" : "households", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : 10,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected People",
     "impactValue" : 25030, 
     "impactUnit" : "households", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Healthcare Infrastructure",
     "impactValue" : 59, 
     "impactUnit" : "Health facilities", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : 10,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Healthcare Infrastructure",
     "impactValue" : 7, 
     "impactUnit" : "Health facilities", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Education Infrastructure",
     "impactValue" : 29, 
     "impactUnit" : "schools", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : 10,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Education Infrastructure",
     "impactValue" : 24, 
     "impactUnit" : "schools", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected Livestock and Animals",
     "impactValue" : 17392, 
     "impactUnit" : "livestocks", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : 10,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected Livestock and Animals",
     "impactValue" : 4824, 
     "impactUnit" : "livestocks", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : 27717, 
     "impactUnit" : "acers of crops", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Other Economic & Livelihood impact",
     "impactValue" : 1300, 
     "impactUnit" : "business", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : 10,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Other Economic & Livelihood impact",
     "impactValue" : 264, 
     "impactUnit" : "business", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : 33, 
     "impactUnit" : "measles cases", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.', 
                     'The cases are drawn from Dadaab (33), Lagdera (22), and Garsen (8) sub-counties from Garissa and Tana River respectively.'],
     "country" : ["Kenya"],
     "location" : ["Dadaab"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : 22, 
     "impactUnit" : "measles cases", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.', 
                     'The cases are drawn from Dadaab (33), Lagdera (22), and Garsen (8) sub-counties from Garissa and Tana River respectively.'],
     "country" : ["Kenya"],
     "location" : ["Lagdera"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : 8, 
     "impactUnit" : "measles cases", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.', 
                     'The cases are drawn from Dadaab (33), Lagdera (22), and Garsen (8) sub-counties from Garissa and Tana River respectively.'],
     "country" : ["Kenya"],
     "location" : ["Garsen"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : 329, 
     "impactUnit" : "cholera cases", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['6 The Ministry of Health indicates that 409 cholera cases have been reported in Lamu (329), Nairobi (49), Tana River (29), Kiambu (1), Isiolo (1) counties.'],
     "country" : ["Kenya"],
     "location" : ["Lamu countie"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : 49, 
     "impactUnit" : "cholera cases", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['6 The Ministry of Health indicates that 409 cholera cases have been reported in Lamu (329), Nairobi (49), Tana River (29), Kiambu (1), Isiolo (1) counties.'],
     "country" : ["Kenya"],
     "location" : ["Nairobi county"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : 29, 
     "impactUnit" : "cholera cases", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['6 The Ministry of Health indicates that 409 cholera cases have been reported in Lamu (329), Nairobi (49), Tana River (29), Kiambu (1), Isiolo (1) counties.'],
     "country" : ["Kenya"],
     "location" : ["Tana River county"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : 1, 
     "impactUnit" : "cholera cases", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['6 The Ministry of Health indicates that 409 cholera cases have been reported in Lamu (329), Nairobi (49), Tana River (29), Kiambu (1), Isiolo (1) counties.'],
     "country" : ["Kenya"],
     "location" : ["Kiambu county"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : 1, 
     "impactUnit" : "cholera cases", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['6 The Ministry of Health indicates that 409 cholera cases have been reported in Lamu (329), Nairobi (49), Tana River (29), Kiambu (1), Isiolo (1) counties.'],
     "country" : ["Kenya"],
     "location" : ["Isiolo county"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 4, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Four fatalities have been reported with a case fatality rate of 0.98%.'],
     "country" : ["Kenya"],
     "location" : ["Nairobi county", "Tana River county", "Kiambu county", "Isiolo county"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    }, 
]

In [83]:
#convert to dataframe
del labelled_reports_dict["appealCode"]
df_list = []
for k,v in labelled_reports_dict.items():
    df = pd.DataFrame(v)
    df['appealCode'] = k
    df_list.append(df)
df_all = pd.concat(df_list)
df_all.reset_index(inplace=True, drop=True)

/var/folders/y5/t1z41tgj7dv50sm2_29dn8740000gp/T/ipykernel_58090/3402577584.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat(df_list)


In [ ]:
#convert to dataframe
del labelled_reports_dict["appealCode"]
df_list = []
for k,v in labelled_reports_dict.items():
    df = pd.DataFrame(v)
    df['appealCode'] = k
    df_list.append(df)
df_all = pd.concat(df_list)
df_all.reset_index(inplace=True, drop=True)

In [28]:
#convert to dataframe
# del labelled_impact_reports_dict["appealCode"]
df_impact_list = []
for k,v in labelled_impact_reports_dict.items():
    df_impact = pd.DataFrame(v)
    df_impact['appealCode'] = k
    df_impact_list.append(df_impact)
df_impact_all = pd.concat(df_impact_list)
df_impact_all.reset_index(inplace=True, drop=True)

In [30]:
#Save impact csv 
fn = "labelled_reports_impact_luca.csv"
df_impact_all.to_csv(DATA_LABELLED+fn, index=False)

In [ ]:
#save
fn = "labelled_reports_luca.csv"
df_all.to_csv(DATA_LABELLED+fn, index=False)